In [1]:
!pip install -q gradio-client pillow pandas huggingface-hub


In [2]:
from huggingface_hub import login

# Login to access gated dataset
login()  # Paste your HF token when prompted


In [3]:
from dataclasses import dataclass

@dataclass
class PipelineConfig:
    # Data
    repo_id: str = "cagliostrolab/860k-ordered-tags"
    shard_index: int = 0
    stage1_csv_path: str = "stage1_index.csv"

    # Processing
    max_images: int = 110
    batch_size: int = 8
    num_workers: int = 4

    # Checkpointing
    checkpoint_interval: int = 100
    checkpoint_dir: str = "./checkpoints"

    # Output
    output_dir: str = "./outputs"

    # Tagger
    tagger_type: str = "deepdanbooru"
    tagger_threshold: float = 0.1
    deepdanbooru_api_timeout: int = 30

    # Validation
    min_confidence_threshold: float = 0.2
    enable_schema_validation: bool = True
    enable_consistency_checks: bool = True
    flag_suspicious_combinations: bool = True

    # Error handling
    save_partial_on_error: bool = True
    error_log_path: str = "./errors.log"

    # NSFW
    process_nsfw: bool = True
    keep_nsfw_flag: bool = True

# Create config instance
config = PipelineConfig(
    shard_index=0,
    max_images=110,
    checkpoint_interval=100
)

print("Config loaded:", config)


Config loaded: PipelineConfig(repo_id='cagliostrolab/860k-ordered-tags', shard_index=0, stage1_csv_path='stage1_index.csv', max_images=110, batch_size=8, num_workers=4, checkpoint_interval=100, checkpoint_dir='./checkpoints', output_dir='./outputs', tagger_type='deepdanbooru', tagger_threshold=0.1, deepdanbooru_api_timeout=30, min_confidence_threshold=0.2, enable_schema_validation=True, enable_consistency_checks=True, flag_suspicious_combinations=True, save_partial_on_error=True, error_log_path='./errors.log', process_nsfw=True, keep_nsfw_flag=True)


In [4]:
from dataclasses import dataclass

@dataclass
class PipelineConfig:
    # Data
    repo_id: str = "cagliostrolab/860k-ordered-tags"
    shard_index: int = 0
    stage1_csv_path: str = "stage1_index.csv"

    # Processing
    max_images: int = 100
    batch_size: int = 8
    num_workers: int = 4

    # Checkpointing
    checkpoint_interval: int = 100
    checkpoint_dir: str = "./checkpoints"

    # Output
    output_dir: str = "./outputs"

    # Tagger
    tagger_type: str = "deepdanbooru"
    tagger_threshold: float = 0.1
    deepdanbooru_api_timeout: int = 30

    # Validation
    min_confidence_threshold: float = 0.2
    enable_schema_validation: bool = True
    enable_consistency_checks: bool = True
    flag_suspicious_combinations: bool = True

    # Error handling
    save_partial_on_error: bool = True
    error_log_path: str = "./errors.log"

    # NSFW
    process_nsfw: bool = True
    keep_nsfw_flag: bool = True

# Create config instance
config = PipelineConfig(
    shard_index=0,
    max_images=100,
    checkpoint_interval=100
)

print("Config loaded:", config)


Config loaded: PipelineConfig(repo_id='cagliostrolab/860k-ordered-tags', shard_index=0, stage1_csv_path='stage1_index.csv', max_images=100, batch_size=8, num_workers=4, checkpoint_interval=100, checkpoint_dir='./checkpoints', output_dir='./outputs', tagger_type='deepdanbooru', tagger_threshold=0.1, deepdanbooru_api_timeout=30, min_confidence_threshold=0.2, enable_schema_validation=True, enable_consistency_checks=True, flag_suspicious_combinations=True, save_partial_on_error=True, error_log_path='./errors.log', process_nsfw=True, keep_nsfw_flag=True)


In [5]:
import re
from typing import Dict, Any, List, Tuple, Optional
from collections import defaultdict

# ===========================
# CONFIGURATION & CONSTANTS
# ===========================

# Competition thresholds
DELTA = 0.3
HIGH_CONF = 0.1
EYE_FALLBACK_THRESHOLD = 0.25

# Limits for multi-value attributes
MAX_HAIR_STYLES = 2
MAX_ACCESSORIES = 5
MAX_CLOTHING_ITEMS = 5
MAX_EXPRESSIONS = 3
MAX_CULTURAL_ELEMENTS = 5

# Character counting
CHAR_COUNT_REGEX = re.compile(r"(\d+)\s*(girl|girls|boy|boys)", re.I)
MULTI_KEYWORDS = {"multiple", "several", "group", "couple", "team"}

# ===========================
# BODY MEASUREMENTS TAGS
# ===========================

# Breast size tags (from Danbooru)
BREAST_SIZE_TAGS = {
    "flat_chest": {"size": "flat", "body_type_weight": {"petite": 0.8, "slim": 0.6}},
    "small_breasts": {"size": "small", "body_type_weight": {"slim": 0.5, "petite": 0.4}},
    "medium_breasts": {"size": "medium", "body_type_weight": {"slim": 0.3, "average": 0.5}},
    "large_breasts": {"size": "large", "body_type_weight": {"curvy": 0.6, "voluptuous": 0.4}},
    "huge_breasts": {"size": "huge", "body_type_weight": {"voluptuous": 0.8, "curvy": 0.5}},
    "gigantic_breasts": {"size": "gigantic", "body_type_weight": {"voluptuous": 0.9}},
}

# Additional breast-related tags
BREAST_DESCRIPTOR_TAGS = {
    "sagging_breasts", "perky_breasts", "bouncing_breasts",
    "hanging_breasts", "asymmetrical_breasts", "oppai_loli"
}

# Hip/Butt size tags
HIP_SIZE_TAGS = {
    "narrow_hips": {"size": "narrow", "body_type_weight": {"slim": 0.6, "petite": 0.5}},
    "wide_hips": {"size": "wide", "body_type_weight": {"curvy": 0.7, "voluptuous": 0.5}},
    "huge_hips": {"size": "huge", "body_type_weight": {"voluptuous": 0.8, "thick": 0.6}},
    "big_ass": {"size": "large", "body_type_weight": {"curvy": 0.6, "thick": 0.5}},
    "huge_ass": {"size": "huge", "body_type_weight": {"voluptuous": 0.7, "thick": 0.6}},
}

# Body proportion tags
BODY_PROPORTION_TAGS = {
    "thick_thighs": {"body_type_weight": {"curvy": 0.5, "thick": 0.6, "athletic": 0.3}},
    "muscular_female": {"body_type_weight": {"muscular": 0.8, "athletic": 0.7}},
    "toned": {"body_type_weight": {"athletic": 0.7, "fit": 0.6}},
    "chubby": {"body_type_weight": {"chubby": 0.8, "plump": 0.6}},
    "skinny": {"body_type_weight": {"slim": 0.7, "skinny": 0.8, "petite": 0.5}},
    "plump": {"body_type_weight": {"plump": 0.7, "chubby": 0.5}},
}

# Combined body type tags (original + inferred)
BODY_TYPE_TAGS = {
    "slim", "slender", "petite", "skinny", "thin",
    "curvy", "voluptuous", "hourglass_figure",
    "muscular", "toned", "athletic", "fit",
    "chubby", "plump", "thick", "heavyset",
    "tall", "short", "average_height"
}

# ===========================
# AGE INFERENCE TAGS
# ===========================

# Age mapping (enhanced)
AGE_TAG_MAP = {
    # Child / pre-adolescent
    "loli": "child",
    "shota": "child",
    "child": "child",
    "preteen": "child",
    "toddler": "child",
    "baby": "child",
    "infant": "child",

    # Teen / adolescent
    "teen": "teen",
    "teenager": "teen",
    "high_schooler": "teen",
    "middle_schooler": "teen",

    # Young adult (late teens into 20s/30s)
    "young_adult": "young adult",
    "college_student": "young adult",
    "adult": "young adult",
    "university_student": "young adult",

    # Middle-aged (established adult)
    "middle_aged": "middle-aged",
    "adult_male": "middle-aged",
    "adult_female": "middle-aged",
    "working_adult": "middle-aged",
    "professional": "middle-aged",
    "parent": "middle-aged",

    # Elderly / seniors
    "elderly": "elderly",
    "old_man": "elderly",
    "old_woman": "elderly",
    "grandfather": "elderly",
    "grandmother": "elderly",
    "aged_up": "elderly",
}

# Age inference signals (multi-modal)
AGE_INFERENCE_SIGNALS = {
    # Clothing-based age hints
    "school_uniform": {"age": "teen", "confidence": 0.7},
    "sailor_uniform": {"age": "teen", "confidence": 0.65},
    "serafuku": {"age": "teen", "confidence": 0.65},
    "gakuran": {"age": "teen", "confidence": 0.6},
    "college_uniform": {"age": "young adult", "confidence": 0.6},
    "business_suit": {"age": "middle-aged", "confidence": 0.5},
    "suit": {"age": "young adult", "confidence": 0.4},
    "wedding_dress": {"age": "young adult", "confidence": 0.5},
    "santa_costume": {"age": "young adult", "confidence": 0.3},

    # Height/proportion hints
    "tall": {"age": "young adult", "confidence": 0.3},
    "short": {"age": "teen", "confidence": 0.25},
    "petite": {"age": "teen", "confidence": 0.3},

    # Facial maturity hints
    "mature": {"age": "middle-aged", "confidence": 0.6},
    "mature_female": {"age": "middle-aged", "confidence": 0.65},
    "milf": {"age": "middle-aged", "confidence": 0.7},
    "mature_male": {"age": "middle-aged", "confidence": 0.65},

    # Body development hints (for anime context)
    "flat_chest": {"age": "teen", "confidence": 0.4},
    "small_breasts": {"age": "teen", "confidence": 0.3},
    "oppai_loli": {"age": "child", "confidence": 0.8},  # specific anime trope
}

# ===========================
# CLOTHING TAGS (SPECIFIC ITEMS)
# ===========================

# Comprehensive clothing item tags (not categories)
SPECIFIC_CLOTHING_TAGS = {
    # Tops
    "shirt", "t-shirt", "polo_shirt", "dress_shirt", "button-up_shirt",
    "blouse", "tank_top", "crop_top", "tube_top", "halter_top",
    "sweater", "cardigan", "hoodie", "pullover", "turtleneck",
    "vest", "waistcoat", "blazer", "jacket", "coat",
    "kimono", "yukata", "hakama", "qipao", "cheongsam",
    "bikini_top", "bra", "sports_bra", "camisole", "bustier",

    # Bottoms
    "pants", "jeans", "trousers", "slacks", "chinos",
    "shorts", "short_shorts", "denim_shorts", "gym_shorts",
    "skirt", "mini_skirt", "pleated_skirt", "pencil_skirt",
    "hakama_skirt", "kilt",

    # Full body
    "dress", "sundress", "cocktail_dress", "evening_gown",
    "wedding_dress", "ballgown", "mini_dress", "maxi_dress",
    "school_uniform", "sailor_uniform", "serafuku", "gakuran",
    "suit", "tuxedo", "business_suit",
    "swimsuit", "one-piece_swimsuit", "bikini",
    "leotard", "bodysuit", "jumpsuit",
    "maid_outfit", "maid_dress", "apron_dress",
    "pajamas", "nightgown", "negligee", "bathrobe",

    # Outerwear
    "trench_coat", "overcoat", "winter_coat", "fur_coat",
    "parka", "windbreaker", "raincoat", "poncho",
    "cape", "cloak", "shawl",

    # Traditional/Cultural
    "hanbok", "sari", "dirndl", "toga", "kesa",

    # Footwear
    "boots", "high_heels", "sandals", "sneakers",
    "loafers", "mary_janes", "platform_shoes",
    "stockings", "thighhighs", "pantyhose", "socks",
    "leg_warmers", "garter_belt",

    # Accessories (wearable)
    "gloves", "mittens", "arm_warmers", "fingerless_gloves",
    "scarf", "necktie", "bowtie", "ribbon", "ascot",
    "belt", "sash", "suspenders",
    "apron", "armband", "wristband",
}

# Eye colors
EYE_COLORS = {
    "blue_eyes", "brown_eyes", "green_eyes", "red_eyes",
    "purple_eyes", "yellow_eyes", "pink_eyes", "orange_eyes",
    "grey_eyes", "heterochromia", "multicolored_eyes"
}

# Hair colors
HAIR_COLOR_TAGS = {
    "black_hair", "brown_hair", "blonde_hair", "white_hair",
    "silver_hair", "grey_hair", "red_hair", "pink_hair",
    "purple_hair", "blue_hair", "green_hair", "orange_hair",
    "multicolored_hair", "gradient_hair", "two-tone_hair"
}

# Hair styles
HAIR_STYLE_TAGS = {
    "ponytail", "twintails", "braid", "twin_braids", "side_braid",
    "bun", "twin_buns", "side_bun", "bob_cut", "hime_cut",
    "straight_hair", "wavy_hair", "curly_hair", "drill_hair",
    "ahoge", "hair_bun", "messy_hair", "spiked_hair",
    "slicked_back_hair", "pixie_cut", "undercut"
}

# Accessories
ACCESSORY_TAGS = {
    # Head / face accessories
    "glasses",
    "sunglasses",
    "eyewear",
    "visor",
    "goggles",
    "hat",
    "beret",
    "cap",
    "headband",
    "hairband",
    "hair_bow",
    "hairclip",
    "hair_ornament",
    "ribbon",
    "tiara",
    "crown",
    "bandana",

    # Jewelry
    "earrings",
    "hoop_earrings",
    "stud_earrings",
    "necklace",
    "choker",
    "pendant",
    "bracelet",
    "anklet",
    "ring",
    "body_chain",
    "brooch",
    "pin",

    # Neck / shoulder
    "scarf",
    "necktie",
    "bowtie",
    "cravat",

    # Bags / carrying
    "backpack",
    "bag",
    "handbag",
    "pouch",
    "satchel",
    "crossbody_bag",
    "fanny_pack",

    # Hand / wrist
    "gloves",
    "fingerless_gloves",
    "armbands",
    "wristbands",
    "watch",

    # Leg / foot
    "stockings",
    "thighhighs",
    "socks",
    "leg_warmers",
    "belt",
    "sash",

    # Misc wearable accessories
    "mask",
    "face_mask",
    "headset",
    "earphones",
    "tie_clip",
    "cufflinks",
    "corset",    # often tagged in outfit contexts
    "vest",      # accessory / layering item
    "cape",
    "shawl",
    "cloak"
}
EXPRESSION_TAGS = {
    # Positive / happy
    "smile",
    "grin",
    "happy",
    "blush",            # often indicates shy / happy
    "smirking",
    "cheerful",
    "gleeful",
    "laughing",
    "content",
    "ecstatic",

    # Neutral / subtle
    "neutral",
    "expressionless",
    "calm",
    "relaxed",
    "serious",
    "poker_face",
    "straight_face",
    "stoic",

    # Negative / sadness
    "sad",
    "tears",
    "crying",
    "depressed",
    "upset",
    "frowning",
    "downcast",
    "mourning",

    # Anger / annoyance
    "angry",
    "annoyed",
    "irritated",
    "scowl",
    "grumpy",
    "shouting",
    "yelling",
    "fierce",
    "rage",

    # Surprise / shock
    "surprised",
    "shocked",
    "astonished",
    "wide_eyes",
    "gasp",

    # Playful / mischievous
    "wink",
    "teasing",
    "mischievous",
    "cheeky",
    "playful",
    "tongue_out",

    # Affection / love
    "love",
    "heart_eyes",
    "affectionate",
    "blowing_kiss",
    "adoring",

    # Embarrassed / shy
    "embarrassed",
    "bashful",
    "flustered",
    "red_face",

    # Other emotion/pose indicators
    "smug",
    "determined",
    "pensive",
    "serene",
    "intense",
    "confident"
}

# Clothing categories (kept for backward compatibility)
CLOTHING_MAP = {
    "uniform": {
        "school_uniform", "sailor_uniform", "military_uniform",
        "gakuran", "serafuku", "summer_uniform", "winter_uniform",
    },
    "traditional": {
        "kimono", "yukata", "hakama", "hanbok", "sari",
        "japanese_clothes", "traditional_clothes", "obi",
        "chinese_clothes", "qipao", "cheongsam"
    },
    "formal": {
        "suit", "tuxedo", "formal_dress", "evening_gown",
        "cocktail_dress", "wedding_dress", "business_suit"
    },
    "casual": {
        "hoodie", "jacket", "blazer", "shirt", "tshirt",
        "crop_top", "tank_top", "hooded_sweater", "sweater",
        "jeans", "shorts", "pants", "denim_jacket", "cardigan"
    },
    "swimwear": {
        "swimsuit", "bikini", "swim_briefs", "string_bikini",
        "sports_bikini", "one-piece_swimsuit"
    },
    "dress": {
        "dress", "sundress", "pleated_skirt", "skirt",
        "micro_dress", "ballgown", "halter_dress", "mini_dress"
    },
    "outerwear": {
        "coat", "trench_coat", "parka", "fur_coat",
        "cape", "cloak", "poncho", "raincoat"
    },
    "footwear": {
        "boots", "sandals", "sneakers", "heels",
        "loafers", "barefoot", "socks", "stockings"
    },
    "costume": {
        "maid_outfit", "cheerleader_uniform", "bunny_costume",
        "cat_costume", "dog_costume", "animal_costume",
        "christmas_costume", "halloween_costume", "cosplay"
    },
    "sleepwear": {
        "pajamas", "nightgown", "negligee", "bathrobe"
    },
}

# Cultural elements mapping
CULTURAL_ELEMENTS_MAP = {
    "japanese_traditional": {
        "kimono", "yukata", "hakama", "japanese_clothes", "obi",
        "geta", "tabi", "fundoshi", "haori"
    },
    "japanese_modern": {
        "school_uniform", "sailor_uniform", "serafuku", "gakuran",
        "meido", "maid", "anime_style"
    },
    "chinese": {
        "qipao", "cheongsam", "chinese_clothes", "mandarin_collar",
        "kung_fu", "hanfu"
    },
    "korean": {
        "hanbok", "korean_clothes", "manhwa", "webtoon"
    },
    "south_asian": {
        "sari", "salwar", "indian_clothes", "bindi", "henna"
    },
    "middle_eastern": {
        "hijab", "abaya", "kaftan", "turban", "arabic_clothes"
    },
    "western_medieval": {
        "armor", "knight", "medieval", "castle", "crown", "royal"
    },
    "western_modern": {
        "suit", "tuxedo", "western_clothes", "cowboy", "jeans"
    },
    "fantasy": {
        "elf", "fairy", "wings", "magic", "wizard", "witch"
    },
    "futuristic": {
        "cyberpunk", "mecha", "sci-fi", "robot", "cyborg", "neon"
    }
}

# Anime origin tags
ANIME_ORIGIN_TAGS = {
    "japanese": {"anime", "manga", "light_novel", "visual_novel", "japanese"},
    "korean": {"manhwa", "webtoon", "korean"},
    "chinese": {"manhua", "donghua", "chinese"},
    "western": {"western", "cartoon", "comic"}
}
# ===========================
# MALE-SPECIFIC BODY TYPE INFERENCE
# ===========================

# Male body measurement tags
MALE_BODY_TAGS = {
    # Muscle definition
    "muscular_male": {"body_type_weight": {"muscular": 0.9, "athletic": 0.6}},
    "muscular": {"body_type_weight": {"muscular": 0.8, "athletic": 0.5}},
    "toned": {"body_type_weight": {"athletic": 0.8, "fit": 0.7}},
    "bara": {"body_type_weight": {"muscular": 0.9, "bulky": 0.7}},  # Heavily muscular

    # Chest/torso development
    "pectorals": {"body_type_weight": {"muscular": 0.6, "athletic": 0.5}},
    "large_pectorals": {"body_type_weight": {"muscular": 0.8, "bulky": 0.6}},
    "abs": {"body_type_weight": {"athletic": 0.7, "fit": 0.6}},
    "six-pack": {"body_type_weight": {"athletic": 0.8, "muscular": 0.6}},

    # Build/frame
    "broad_shoulders": {"body_type_weight": {"muscular": 0.6, "athletic": 0.5}},
    "thick_arms": {"body_type_weight": {"muscular": 0.7, "athletic": 0.4}},
    "thick_neck": {"body_type_weight": {"muscular": 0.6, "bulky": 0.5}},
    "strongman_waist": {"body_type_weight": {"bulky": 0.8, "muscular": 0.5}},

    # Slim/lean builds
    "skinny_male": {"body_type_weight": {"slim": 0.8, "skinny": 0.9}},
    "thin": {"body_type_weight": {"slim": 0.7, "skinny": 0.6}},
    "slender": {"body_type_weight": {"slim": 0.6, "lean": 0.5}},
    "lean": {"body_type_weight": {"athletic": 0.5, "slim": 0.4}},

    # Heavy/larger builds
    "chubby_male": {"body_type_weight": {"chubby": 0.8, "heavyset": 0.5}},
    "fat_man": {"body_type_weight": {"heavyset": 0.9, "obese": 0.7}},
    "belly": {"body_type_weight": {"heavyset": 0.6, "chubby": 0.5}},
    "dad_bod": {"body_type_weight": {"average": 0.6, "chubby": 0.4}},
    "beer_belly": {"body_type_weight": {"chubby": 0.7, "heavyset": 0.5}},

    # Height/stature
    "tall_male": {"body_type_weight": {"tall": 0.7}},
    "short_male": {"body_type_weight": {"short": 0.7}},
    "giant": {"body_type_weight": {"tall": 0.9, "bulky": 0.5}},

    # Special builds
    "otoko_no_ko": {"body_type_weight": {"slim": 0.7, "petite": 0.5}},
    #Feminine male
    "femboy": {"body_type_weight": {"slim": 0.8, "petite": 0.6}},
    "trap_(gender)": {"body_type_weight": {"slim": 0.7, "petite": 0.5}},
}

# Male body hair (secondary characteristic for body type + age) [web:26]
MALE_BODY_HAIR_TAGS = {
    "chest_hair": {"body_type_weight": {"muscular": 0.3, "mature": 0.4}, "age_hint": "middle-aged"},
    "arm_hair": {"body_type_weight": {"muscular": 0.2, "mature": 0.3}, "age_hint": "young adult"},
    "leg_hair": {"body_type_weight": {"athletic": 0.2}, "age_hint": "young adult"},
    "hairy": {"body_type_weight": {"mature": 0.5}, "age_hint": "middle-aged"},
    "hairy_male": {"body_type_weight": {"mature": 0.6}, "age_hint": "middle-aged"},
    "body_hair": {"body_type_weight": {"mature": 0.4}, "age_hint": "young adult"},
}

# ===========================
# MALE-SPECIFIC AGE INFERENCE
# ===========================

# Facial hair as strong age indicator [web:26][web:16]
MALE_FACIAL_HAIR_AGE = {
    "stubble": {"age": "young adult", "confidence": 0.6},
    "5_o'clock_shadow": {"age": "young adult", "confidence": 0.5},
    "beard": {"age": "middle-aged", "confidence": 0.7},
    "full_beard": {"age": "middle-aged", "confidence": 0.75},
    "goatee": {"age": "young adult", "confidence": 0.55},
    "mustache": {"age": "middle-aged", "confidence": 0.65},
    "long_beard": {"age": "elderly", "confidence": 0.8},
    "white_beard": {"age": "elderly", "confidence": 0.85},
    "sideburns": {"age": "young adult", "confidence": 0.4},
    "soul_patch": {"age": "young adult", "confidence": 0.5},
}

# Male-specific facial features for age [web:16]
MALE_FACIAL_AGE_HINTS = {
    # Youth markers
    "round_face": {"age": "teen", "confidence": 0.5},
    "soft_features": {"age": "teen", "confidence": 0.4},
    "large_eyes": {"age": "teen", "confidence": 0.3},
    "smooth_skin": {"age": "teen", "confidence": 0.4},

    # Young adult markers (20s-30s)
    "sharp_jawline": {"age": "young adult", "confidence": 0.5},
    "defined_jawline": {"age": "young adult", "confidence": 0.5},
    "chiseled_features": {"age": "young adult", "confidence": 0.6},
    "narrow_eyes": {"age": "young adult", "confidence": 0.3},

    # Middle-age markers (40s-50s)
    "wrinkles": {"age": "middle-aged", "confidence": 0.7},
    "eye_wrinkles": {"age": "middle-aged", "confidence": 0.65},
    "forehead_wrinkles": {"age": "middle-aged", "confidence": 0.6},
    "defined_cheekbones": {"age": "middle-aged", "confidence": 0.5},
    "mature_male": {"age": "middle-aged", "confidence": 0.8},
    "dilf": {"age": "middle-aged", "confidence": 0.85},

    # Elderly markers (60+)
    "receding_hairline": {"age": "middle-aged", "confidence": 0.6},
    "bald": {"age": "middle-aged", "confidence": 0.5},
    "bald_male": {"age": "middle-aged", "confidence": 0.55},
    "grey_hair": {"age": "elderly", "confidence": 0.7},
    "white_hair": {"age": "elderly", "confidence": 0.6},
    "sagging_skin": {"age": "elderly", "confidence": 0.8},
    "old_man": {"age": "elderly", "confidence": 0.95},
    "elderly_male": {"age": "elderly", "confidence": 0.9},
    "grandfather": {"age": "elderly", "confidence": 0.85},
}

# Male-specific clothing age hints [web:26]
MALE_CLOTHING_AGE_HINTS = {
    # Youth/Teen
    "gakuran": {"age": "teen", "confidence": 0.7},
    "school_uniform": {"age": "teen", "confidence": 0.65},
    "gym_uniform": {"age": "teen", "confidence": 0.6},
    "track_suit": {"age": "teen", "confidence": 0.5},
    "hoodie": {"age": "teen", "confidence": 0.35},
    "backpack": {"age": "teen", "confidence": 0.4},

    # Young adult
    "tuxedo": {"age": "young adult", "confidence": 0.6},
    "business_suit": {"age": "young adult", "confidence": 0.55},
    "dress_shirt": {"age": "young adult", "confidence": 0.4},
    "necktie": {"age": "young adult", "confidence": 0.45},
    "casual_suit": {"age": "young adult", "confidence": 0.5},

    # Middle-aged
    "salaryman": {"age": "middle-aged", "confidence": 0.75},
    "businessman": {"age": "middle-aged", "confidence": 0.7},
    "three_piece_suit": {"age": "middle-aged", "confidence": 0.65},
    "suspenders": {"age": "middle-aged", "confidence": 0.5},
    "vest": {"age": "middle-aged", "confidence": 0.4},

    # Elderly
    "traditional_clothes": {"age": "elderly", "confidence": 0.4},
    "kimono": {"age": "middle-aged", "confidence": 0.3},
    "walking_stick": {"age": "elderly", "confidence": 0.8},
    "cane": {"age": "elderly", "confidence": 0.8},
}

# Male body build age correlation
MALE_BUILD_AGE_HINTS = {
    # Skinny/underdeveloped = younger
    "skinny": {"age": "teen", "confidence": 0.3},
    "thin": {"age": "teen", "confidence": 0.25},

    # Athletic/muscular = young adult prime
    "muscular": {"age": "young adult", "confidence": 0.4},
    "athletic": {"age": "young adult", "confidence": 0.45},
    "toned": {"age": "young adult", "confidence": 0.4},

    # Dad bod/belly = middle-aged
    "dad_bod": {"age": "middle-aged", "confidence": 0.7},
    "beer_belly": {"age": "middle-aged", "confidence": 0.65},
    "belly": {"age": "middle-aged", "confidence": 0.4},
}

# ===========================
# MALE-SPECIFIC HAIR STYLES
# ===========================

# Male hair style tags [web:26][web:29]
MALE_HAIR_STYLE_TAGS = {
    # Very short
    "bald": {"style": "bald", "length": "none"},
    "buzz_cut": {"style": "buzz cut", "length": "very_short"},
    "crew_cut": {"style": "crew cut", "length": "very_short"},
    "flat_top": {"style": "flat top", "length": "very_short"},

    # Short styles
    "short_hair": {"style": "short", "length": "short"},
    "undercut": {"style": "undercut", "length": "short"},
    "sidecut": {"style": "sidecut", "length": "short"},
    "fade": {"style": "fade", "length": "short"},
    "military_cut": {"style": "military cut", "length": "very_short"},

    # Medium/styled
    "pompadour": {"style": "pompadour", "length": "medium"},
    "slicked_back_hair": {"style": "slicked back", "length": "medium"},
    "hair_slicked_back": {"style": "slicked back", "length": "medium"},
    "quiff": {"style": "quiff", "length": "medium"},
    "curtained_hair": {"style": "curtains", "length": "medium"},
    "middle_part": {"style": "middle part", "length": "medium"},
    "side_part": {"style": "side part", "length": "short"},

    # Edgy/alternative
    "mohawk": {"style": "mohawk", "length": "short"},
    "spiky_hair": {"style": "spiky", "length": "short"},
    "spiked_hair": {"style": "spiked", "length": "short"},
    "faux_hawk": {"style": "faux hawk", "length": "short"},
    "dreadlocks": {"style": "dreadlocks", "length": "long"},

    # Long styles (less common for males)
    "long_hair": {"style": "long", "length": "long"},
    "ponytail": {"style": "ponytail", "length": "long"},
    "man_bun": {"style": "man bun", "length": "medium"},
    "topknot": {"style": "topknot", "length": "medium"},
    "samurai_topknot": {"style": "samurai topknot", "length": "medium"},

    # Messy/natural
    "messy_hair": {"style": "messy", "length": "varies"},
    "unkempt_hair": {"style": "unkempt", "length": "varies"},
    "shaggy_hair": {"style": "shaggy", "length": "medium"},
    "tousled_hair": {"style": "tousled", "length": "medium"},

    # Facial hair (part of overall hairstyle)
    "sideburns": {"style": "sideburns", "length": "short"},
    "mutton_chops": {"style": "mutton chops", "length": "medium"},
}

# ===========================
# MALE INFERENCE FUNCTIONS
# ===========================

def _infer_male_body_type(tag_probs: Dict[str, float]) -> Tuple[str, float, List[str]]:
    """
    Male-specific body type inference using muscle definition, build, and body hair.
    Returns: (body_type, confidence, evidence_list)
    """
    body_type_scores = defaultdict(float)
    evidence = []

    # Step 1: Check male-specific body tags
    for tag, data in MALE_BODY_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{tag}:{prob:.2f}")

    # Step 2: Check body hair (maturity indicator)
    for tag, data in MALE_BODY_HAIR_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight * 0.5  # Lower weight
            evidence.append(f"{tag}:{prob:.2f}(hair)")

    # Step 3: Fallback to general body tags
    for body_tag in BODY_TYPE_TAGS:
        if body_tag in tag_probs and tag_probs[body_tag] > HIGH_CONF:
            prob = tag_probs[body_tag]
            body_type_scores[body_tag] += prob * 1.2
            evidence.append(f"{body_tag}:{prob:.2f}(general)")

    if not body_type_scores:
        return "unknown", 1.0, []

    body_type, conf = _resolve_competition(dict(body_type_scores))

    return body_type, conf, evidence[:5]


def _infer_male_age(
    tag_probs: Dict[str, float],
    body_type: str,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    Male-specific age inference using:
    - Direct age tags
    - Facial hair (STRONG signal for males)
    - Facial features (jawline, wrinkles)
    - Body build maturity
    - Clothing context

    Returns: (age, confidence, evidence_list)
    """
    age_scores = defaultdict(float)
    evidence = []

    # Step 1: Direct age tags (highest weight)
    for tag, mapped_age in AGE_TAG_MAP.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[mapped_age] += prob * 2.0
            evidence.append(f"direct:{tag}:{prob:.2f}")

    # Step 2: Facial hair (VERY strong signal for males)
    for tag, signal in MALE_FACIAL_HAIR_AGE.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"] * 1.5  # High weight
            evidence.append(f"facial_hair:{tag}:{prob:.2f}")

    # Step 3: Facial features
    for tag, signal in MALE_FACIAL_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"facial:{tag}:{prob:.2f}")

    # Step 4: Clothing-based age hints
    for item in clothing_items:
        item_tag = item.replace(" ", "_")
        if item_tag in MALE_CLOTHING_AGE_HINTS:
            signal = MALE_CLOTHING_AGE_HINTS[item_tag]
            age_scores[signal["age"]] += signal["confidence"]
            evidence.append(f"clothing:{item}:{signal['confidence']:.2f}")

    # Step 5: Additional clothing tags not in items list
    for tag, signal in MALE_CLOTHING_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"clothing:{tag}:{prob:.2f}")

    # Step 6: Body build age correlation
    for tag, signal in MALE_BUILD_AGE_HINTS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"build:{tag}:{prob:.2f}")

    # Step 7: Body hair as age hint
    for tag, data in MALE_BODY_HAIR_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            if "age_hint" in data:
                age_scores[data["age_hint"]] += prob * 0.4
                evidence.append(f"body_hair:{tag}:{prob:.2f}")

    if not age_scores:
        return "unknown", 1.0, []

    age, conf = _resolve_competition(dict(age_scores))

    # Normalize confidence
    max_possible_score = 4.0  # Adjusted for male-specific signals
    normalized_conf = min(conf / max_possible_score, 1.0)

    return age, normalized_conf, evidence[:7]


def _extract_male_hair_styles(tag_probs: Dict[str, float]) -> List[Dict[str, Any]]:
    """
    Extract male-specific hair styles with length information.
    Returns list of {value, confidence, length} dictionaries.
    """
    hair_styles = []

    for tag, data in MALE_HAIR_STYLE_TAGS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            hair_styles.append({
                "value": data["style"],
                "confidence": round(tag_probs[tag], 3),
                "length": data["length"]
            })

    # Sort by confidence
    hair_styles.sort(key=lambda x: x["confidence"], reverse=True)

    return hair_styles[:MAX_HAIR_STYLES]


def infer_body_type_gender_aware(
    tag_probs: Dict[str, float],
    gender: str
) -> Tuple[str, float, List[str]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific body type inference.
    """
    if gender == "male":
        return _infer_male_body_type(tag_probs)
    else:  # female or unknown
        return _infer_body_type_from_measurements(tag_probs)


def infer_age_gender_aware(
    tag_probs: Dict[str, float],
    gender: str,
    body_type: str,
    breast_size: Optional[str] = None,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific age inference.
    """
    if gender == "male":
        return _infer_male_age(tag_probs, body_type, clothing_items)
    else:  # female
        return _infer_age_multimodal(tag_probs, gender, breast_size, clothing_items)


def extract_hair_styles_gender_aware(
    tag_probs: Dict[str, float],
    gender: str
) -> List[Dict[str, Any]]:
    """
    HOT-SWAPPABLE: Routes to gender-specific hair style extraction.
    """
    if gender == "male":
        return _extract_male_hair_styles(tag_probs)
    else:  # female or unknown - use general tags
        hair_style_tags = {t for t in tag_probs if t in HAIR_STYLE_TAGS}
        return _select_top_n(tag_probs, hair_style_tags, MAX_HAIR_STYLES)


# ===========================
# HELPER FUNCTIONS
# ===========================

def estimate_character_count(tags: List[str]) -> Tuple[int, bool]:
    """
    Estimates character count from DeepDanbooru tags.
    Returns: (count, is_ambiguous)
    """
    total_count = 0
    has_solo = False
    has_multi_keyword = False

    for tag in tags:
        tag_clean = tag.lower().strip()

        if tag_clean == "solo":
            has_solo = True

        match = CHAR_COUNT_REGEX.match(tag_clean)
        if match:
            total_count += int(match.group(1))

        if any(keyword in tag_clean for keyword in MULTI_KEYWORDS):
            has_multi_keyword = True

    # Decision logic
    if has_solo:
        return 1, False

    if total_count == 0:
        return (2, True) if has_multi_keyword else (0, False)

    if total_count == 1:
        return (2, True) if has_multi_keyword else (1, True)

    # total_count >= 2
    return (total_count + 2, True) if has_multi_keyword else (total_count, True)


def _resolve_competition(candidates: Dict[str, float]) -> Tuple[str, float]:
    """
    Resolves competition between candidate attributes.
    Returns: (winner, confidence) or ("unknown", 1.0) if no candidates
    """
    if not candidates:
        return "unknown", 1.0

    # Filter out zero probabilities
    candidates = {k: v for k, v in candidates.items() if v > 0}

    if not candidates:
        return "unknown", 1.0

    sorted_items = sorted(candidates.items(), key=lambda x: x[1], reverse=True)

    if len(sorted_items) == 1:
        return sorted_items[0]

    (v1, p1), (v2, p2) = sorted_items[:2]

    # If top two are very close, return ambiguous
    if abs(p1 - p2) < DELTA:
        return v1, p1

    return v1, p1


def _infer_body_type_from_measurements(
    tag_probs: Dict[str, float]
) -> Tuple[str, float, List[str]]:
    """
    Enhanced body type inference using breast size, hip size, and body proportions.
    Returns: (body_type, confidence, evidence_list)
    """
    body_type_scores = defaultdict(float)
    evidence = []

    # Step 1: Check breast size tags
    for breast_tag, data in BREAST_SIZE_TAGS.items():
        if breast_tag in tag_probs and tag_probs[breast_tag] > HIGH_CONF:
            prob = tag_probs[breast_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{breast_tag}:{prob:.2f}")

    # Step 2: Check hip/butt size tags
    for hip_tag, data in HIP_SIZE_TAGS.items():
        if hip_tag in tag_probs and tag_probs[hip_tag] > HIGH_CONF:
            prob = tag_probs[hip_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{hip_tag}:{prob:.2f}")

    # Step 3: Check body proportion tags
    for prop_tag, data in BODY_PROPORTION_TAGS.items():
        if prop_tag in tag_probs and tag_probs[prop_tag] > HIGH_CONF:
            prob = tag_probs[prop_tag]
            for body_type, weight in data["body_type_weight"].items():
                body_type_scores[body_type] += prob * weight
            evidence.append(f"{prop_tag}:{prob:.2f}")

    # Step 4: Check direct body type tags
    for body_tag in BODY_TYPE_TAGS:
        if body_tag in tag_probs and tag_probs[body_tag] > HIGH_CONF:
            prob = tag_probs[body_tag]
            body_type_scores[body_tag] += prob * 1.5  # Higher weight for direct tags
            evidence.append(f"{body_tag}:{prob:.2f}(direct)")

    # Resolve competition
    if not body_type_scores:
        return "unknown", 1.0, []

    body_type, conf = _resolve_competition(dict(body_type_scores))

    return body_type, conf, evidence[:5]  # Limit evidence to top 5


def _infer_age_multimodal(
    tag_probs: Dict[str, float],
    gender: str,
    breast_size: Optional[str] = None,
    clothing_items: List[str] = []
) -> Tuple[str, float, List[str]]:
    """
    Multi-modal age inference using:
    - Direct age tags
    - Clothing context (school uniform = teen)
    - Body development (breast size for female characters)
    - Height/proportion hints
    - Facial maturity tags

    Returns: (age, confidence, evidence_list)
    """
    age_scores = defaultdict(float)
    evidence = []

    # Step 1: Direct age tags (highest weight)
    for tag, mapped_age in AGE_TAG_MAP.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[mapped_age] += prob * 2.0  # Strong signal
            evidence.append(f"direct:{tag}:{prob:.2f}")

    # Step 2: Clothing-based age hints
    for item in clothing_items:
        if item in AGE_INFERENCE_SIGNALS:
            signal = AGE_INFERENCE_SIGNALS[item]
            age_scores[signal["age"]] += signal["confidence"]
            evidence.append(f"clothing:{item}:{signal['confidence']:.2f}")

    # Step 3: Additional contextual tags
    for tag, signal in AGE_INFERENCE_SIGNALS.items():
        if tag in tag_probs and tag_probs[tag] > HIGH_CONF:
            prob = tag_probs[tag]
            age_scores[signal["age"]] += prob * signal["confidence"]
            evidence.append(f"context:{tag}:{prob:.2f}")

    # Step 4: Breast size inference (anime-specific, for female characters)
    if gender == "female" and breast_size:
        if breast_size in ["flat", "small"]:
            age_scores["teen"] += 0.4
            evidence.append(f"breast_size:{breast_size}:0.40")
        elif breast_size == "oppai_loli":  # Specific anime trope
            age_scores["child"] += 0.8
            evidence.append(f"breast_size:oppai_loli:0.80")
        elif breast_size in ["huge", "gigantic"]:
            age_scores["young adult"] += 0.3
            age_scores["middle-aged"] += 0.2
            evidence.append(f"breast_size:{breast_size}:0.30")

    # Step 5: Height/stature inference
    if "tall" in tag_probs and tag_probs["tall"] > HIGH_CONF:
        age_scores["young adult"] += 0.3
        evidence.append(f"height:tall:{tag_probs['tall']:.2f}")
    elif "petite" in tag_probs and tag_probs["petite"] > HIGH_CONF:
        age_scores["teen"] += 0.3
        evidence.append(f"height:petite:{tag_probs['petite']:.2f}")

    # Resolve competition
    if not age_scores:
        return "unknown", 1.0, []

    age, conf = _resolve_competition(dict(age_scores))

    # Normalize confidence to 0-1 range
    max_possible_score = 3.0  # Rough estimate of max accumulation
    normalized_conf = min(conf / max_possible_score, 1.0)

    return age, normalized_conf, evidence[:7]  # Top 7 evidence items


def _extract_specific_clothing(tag_probs: Dict[str, float]) -> List[Dict[str, Any]]:
    """
    Extract specific clothing items (not categories) from tags.
    Returns list of {item, confidence} dictionaries.
    """
    clothing_items = []

    for tag, prob in tag_probs.items():
        # Check if tag is a specific clothing item
        if tag in SPECIFIC_CLOTHING_TAGS and prob > HIGH_CONF:
            clothing_items.append({
                "item": tag.replace("_", " "),
                "confidence": round(prob, 3)
            })

    # Sort by confidence
    clothing_items.sort(key=lambda x: x["confidence"], reverse=True)

    return clothing_items[:MAX_CLOTHING_ITEMS]


def _select_top_n(tag_probs: Dict[str, float], valid_tags: set, n: int) -> List[Dict[str, float]]:
    """Select top N tags from valid set, with confidence scores"""
    filtered = [(t, p) for t, p in tag_probs.items() if t in valid_tags and p > 0]
    filtered.sort(key=lambda x: x[1], reverse=True)
    return [{"value": t, "confidence": p} for t, p in filtered[:n]]


def _infer_skin_tone_from_context(
    tag_probs: Dict[str, float],
    hair_color: str,
    eye_color: str
) -> Tuple[str, float]:
    """
    Infer skin tone from hair and eye color when direct tags are missing.
    Uses common anime art style associations.
    """
    # Common associations in anime
    inference_map = {
        ("blonde", "blue"): ("light", 0.4),
        ("blonde", "green"): ("light", 0.4),
        ("silver", "blue"): ("very_light", 0.4),
        ("white", "red"): ("very_light", 0.4),
        ("black", "brown"): ("medium", 0.35),
        ("black", "black"): ("medium", 0.35),
        ("brown", "brown"): ("light", 0.35),
        ("red", "green"): ("light", 0.3),
    }

    key = (hair_color, eye_color)
    if key in inference_map:
        return inference_map[key]

    # Fallback: very weak inference
    if hair_color == "black":
        return "medium", 0.25
    elif hair_color in ["blonde", "silver", "white"]:
        return "light", 0.25

    return "unknown", 1.0


def _analyze_cultural_elements(tag_probs: Dict[str, float]) -> List[Dict[str, float]]:
    """Identify cultural elements from clothing and setting tags"""
    cultural_scores = {}

    for culture, tags in CULTURAL_ELEMENTS_MAP.items():
        score = max([tag_probs.get(t, 0.0) for t in tags], default=0.0)
        if score > HIGH_CONF:
            cultural_scores[culture] = score

    # Sort and return top N
    sorted_cultures = sorted(cultural_scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {"element": culture, "confidence": conf}
        for culture, conf in sorted_cultures[:MAX_CULTURAL_ELEMENTS]
    ]


def _estimate_ethnicity(
    tag_probs: Dict[str, float],
    skin_tone: str,
    hair_color: str,
    eye_color: str,
    cultural_elements: List[Dict[str, float]]
) -> Dict[str, Any]:
    """
    Probabilistic ethnicity estimation using multi-signal fusion.
    IMPORTANT: This is a statistical estimate, not definitive identification.
    """
    ethnicity_scores = defaultdict(float)
    signals_used = []

    # Strong signal: Cultural clothing (weight: 0.6)
    for elem in cultural_elements:
        culture = elem["element"]
        conf = elem["confidence"]

        if "japanese" in culture:
            ethnicity_scores["east_asian"] += conf * 0.6
            signals_used.append(f"clothing:{culture}")
        elif "chinese" in culture:
            ethnicity_scores["east_asian"] += conf * 0.6
            signals_used.append(f"clothing:{culture}")
        elif "korean" in culture:
            ethnicity_scores["east_asian"] += conf * 0.6
            signals_used.append(f"clothing:{culture}")
        elif "south_asian" in culture:
            ethnicity_scores["south_asian"] += conf * 0.6
            signals_used.append(f"clothing:{culture}")
        elif "middle_eastern" in culture:
            ethnicity_scores["middle_eastern"] += conf * 0.6
            signals_used.append(f"clothing:{culture}")
        elif "western" in culture:
            ethnicity_scores["caucasian"] += conf * 0.4
            signals_used.append(f"clothing:{culture}")

    # Medium signal: Skin tone (weight: 0.3)
    if skin_tone in ["very_light", "light"]:
        ethnicity_scores["east_asian"] += 0.3
        ethnicity_scores["caucasian"] += 0.3
        signals_used.append(f"skin:{skin_tone}")
    elif skin_tone in ["tan", "medium", "olive"]:
        ethnicity_scores["east_asian"] += 0.2
        ethnicity_scores["south_asian"] += 0.2
        ethnicity_scores["latin_american"] += 0.2
        signals_used.append(f"skin:{skin_tone}")
    elif skin_tone in ["brown", "dark", "very_dark"]:
        ethnicity_scores["african"] += 0.3
        ethnicity_scores["south_asian"] += 0.2
        signals_used.append(f"skin:{skin_tone}")

    # Weak signal: Hair + Eye combo (weight: 0.2)
    if hair_color == "black" and eye_color in ["brown", "black"]:
        ethnicity_scores["east_asian"] += 0.2
        ethnicity_scores["south_asian"] += 0.1
        signals_used.append(f"hair:black+eyes:{eye_color}")
    elif hair_color in ["blonde", "silver"] and eye_color == "blue":
        ethnicity_scores["caucasian"] += 0.2
        signals_used.append(f"hair:{hair_color}+eyes:blue")

    # Normalize scores to 0-1 range
    if ethnicity_scores:
        max_score = max(ethnicity_scores.values())
        if max_score > 0:
            ethnicity_scores = {k: v/max_score for k, v in ethnicity_scores.items()}

    if not ethnicity_scores:
        return {
            "primary": "unknown",
            "confidence": 1.0,
            "signals_used": [],
            "disclaimer": "insufficient data for estimation"
        }

    primary = max(ethnicity_scores.items(), key=lambda x: x[1])

    return {
        "primary": primary[0],
        "confidence": round(primary[1], 2),
        "signals_used": signals_used[:5],
        "disclaimer": "probabilistic estimate from visual features, not definitive"
    }


def _determine_anime_origin(tag_probs: Dict[str, float]) -> Tuple[str, float]:
    """Determine the origin/style of the anime content"""
    origin_scores = {}

    for origin, tags in ANIME_ORIGIN_TAGS.items():
        score = max([tag_probs.get(t, 0.0) for t in tags], default=0.0)
        origin_scores[origin] = score

    # Default to japanese if no specific tags
    if not any(score > 0 for score in origin_scores.values()):
        return "japanese", 0.7

    origin, conf = max(origin_scores.items(), key=lambda x: x[1])
    return origin, conf


def _generate_scene_description(tag_probs: Dict[str, float]) -> Dict[str, Any]:
    """Generate description for images without characters"""
    sorted_tags = sorted(tag_probs.items(), key=lambda x: x[1], reverse=True)[:10]

    scene_type = "unknown"
    if any(t in tag_probs for t in ["landscape", "scenery", "outdoors", "nature"]):
        scene_type = "landscape"
    elif any(t in tag_probs for t in ["vehicle", "car", "mecha", "robot"]):
        scene_type = "vehicle"
    elif any(t in tag_probs for t in ["building", "architecture", "city", "urban"]):
        scene_type = "architecture"
    elif any(t in tag_probs for t in ["object", "still_life", "food"]):
        scene_type = "object"
    elif any(t in tag_probs for t in ["abstract", "pattern", "texture"]):
        scene_type = "abstract"

    return {
        "image_status": "no_character_detected",
        "scene_type": scene_type,
        "top_tags": [{"tag": t, "confidence": round(c, 3)} for t, c in sorted_tags]
    }


# ===========================
# MAIN PROJECTOR
# ===========================

def run_projector(tag_probs: Dict[str, float]) -> Dict[str, Any]:
    """
    Enhanced gender-aware projector using inference modules.
    """
    tag_list = list(tag_probs.keys())

    # Character count gate
    count, ambiguous = estimate_character_count(tag_list)

    if count == 0:
        return _generate_scene_description(tag_probs)

    if count != 1:
        return {
            "image_status": "ambiguous_multi_character",
            "character_count": count,
            "attributes": None
        }

    output = {"image_status": "single_character"}

    # ---- Gender (FIRST - needed for routing) ----
    gender_candidates = {
        "female": tag_probs.get("1girl", 0.0),
        "male": tag_probs.get("1boy", 0.0)
    }
    gender, g_conf = _resolve_competition(gender_candidates)
    output["gender"] = gender
    output["gender_confidence"] = round(g_conf, 3)

    # ---- Eye Color ----
    eye_candidates = {
        t.replace("_eyes", ""): p
        for t, p in tag_probs.items()
        if t in EYE_COLORS and p >= HIGH_CONF
    }
    if not eye_candidates:
        eye_candidates = {
            t.replace("_eyes", ""): p
            for t, p in tag_probs.items()
            if t in EYE_COLORS and p >= EYE_FALLBACK_THRESHOLD
        }
    eye_color, e_conf = _resolve_competition(eye_candidates)
    output["eye_color"] = eye_color
    output["eye_color_confidence"] = round(e_conf, 3)

    # ---- Hair Color ----
    hair_color_candidates = {
        color.replace("_hair", ""): tag_probs.get(color, 0.0)
        for color in HAIR_COLOR_TAGS
        if color in tag_probs
    }
    hair_color, hc_conf = _resolve_competition(hair_color_candidates)
    output["hair_color"] = hair_color
    output["hair_color_confidence"] = round(hc_conf, 3)

    # ---- Hair Length ----
    hair_length_candidates = {
        "long": tag_probs.get("long_hair", 0.0),
        "short": tag_probs.get("short_hair", 0.0),
        "medium": tag_probs.get("medium_hair", 0.0),
        "shoulder": tag_probs.get("shoulder_length_hair", 0.0),
        "midback": tag_probs.get("midback_length_hair", 0.0),
        "waist": tag_probs.get("waist_length_hair", 0.0),
        "hip": tag_probs.get("hip_length_hair", 0.0),
        "ankle": tag_probs.get("ankle_length_hair", 0.0)
    }
    hair_length, hl_conf = _resolve_competition(hair_length_candidates)
    output["hair_length"] = hair_length
    output["hair_length_confidence"] = round(hl_conf, 3)

    # ---- Hair Styles (GENDER-AWARE) ----
    hair_styles = extract_hair_styles_gender_aware(tag_probs, gender)
    output["hair_styles"] = hair_styles

    # ---- Skin Tone ----
    skin_candidates = {
        "very_light": tag_probs.get("pale_skin", 0.0),
        "light": tag_probs.get("light_skin", 0.0),
        "tan": tag_probs.get("tan_skin", 0.0),
        "medium": tag_probs.get("medium_skin", 0.0),
        "olive": tag_probs.get("olive_skin", 0.0),
        "brown": tag_probs.get("brown_skin", 0.0),
        "dark": tag_probs.get("dark_skin", 0.0),
        "very_dark": tag_probs.get("very_dark_skin", 0.0),
        "colored": tag_probs.get("colored_skin", 0.0),
    }
    skin, s_conf = _resolve_competition(skin_candidates)
    if skin == "unknown":
        skin, s_conf = _infer_skin_tone_from_context(tag_probs, hair_color, eye_color)
    output["skin_tone"] = skin
    output["skin_tone_confidence"] = round(s_conf, 3)

    # ---- Body Type (GENDER-AWARE) ----
    body_type, bt_conf, body_evidence = infer_body_type_gender_aware(tag_probs, gender)
    output["body_type"] = body_type
    output["body_type_confidence"] = round(bt_conf, 3)
    output["body_type_evidence"] = body_evidence

    # ---- Extract breast size (for female age inference) ----
    breast_size = None
    if gender == "female":
        for breast_tag, data in BREAST_SIZE_TAGS.items():
            if breast_tag in tag_probs and tag_probs[breast_tag] > HIGH_CONF:
                breast_size = data["size"]
                break

    # ---- Clothing Items ----
    specific_clothing = _extract_specific_clothing(tag_probs)
    output["clothing_items"] = specific_clothing

    clothing_categories = []
    for category, tags in CLOTHING_MAP.items():
        max_score = max([tag_probs.get(t, 0.0) for t in tags], default=0.0)
        if max_score > HIGH_CONF:
            clothing_categories.append({"category": category, "confidence": round(max_score, 3)})
    clothing_categories.sort(key=lambda x: x["confidence"], reverse=True)
    output["clothing"] = clothing_categories[:MAX_CLOTHING_ITEMS]

    # ---- Age (GENDER-AWARE) ----
    clothing_item_names = [item["item"].replace(" ", "_") for item in specific_clothing]
    age, age_conf, age_evidence = infer_age_gender_aware(
        tag_probs, gender, body_type, breast_size, clothing_item_names
    )
    output["age"] = age
    output["age_confidence"] = round(age_conf, 3)
    output["age_evidence"] = age_evidence

    # ---- Expression ----
    expression_candidates = {
        t: p for t, p in tag_probs.items()
        if t in EXPRESSION_TAGS and p >= HIGH_CONF
    }
    if expression_candidates:
        sorted_expressions = sorted(expression_candidates.items(), key=lambda x: x[1], reverse=True)
        output["expressions"] = [
            {"expression": expr, "confidence": round(conf, 3)}
            for expr, conf in sorted_expressions[:MAX_EXPRESSIONS]
        ]
    else:
        output["expressions"] = [{"expression": "unknown", "confidence": 1.0}]

    # ---- Accessories ----
    accessory_tags = {t for t in tag_probs if t in ACCESSORY_TAGS}
    accessories = _select_top_n(tag_probs, accessory_tags, MAX_ACCESSORIES)
    output["accessories"] = accessories

    # ---- Cultural Elements ----
    cultural_elements = _analyze_cultural_elements(tag_probs)
    output["cultural_elements"] = cultural_elements

    # ---- Ethnicity Estimation ----
    ethnicity_data = _estimate_ethnicity(
        tag_probs, skin, hair_color, eye_color, cultural_elements
    )
    output["ethnicity_estimate"] = ethnicity_data

    # ---- Anime Origin ----
    anime_origin, ao_conf = _determine_anime_origin(tag_probs)
    output["anime_origin"] = anime_origin
    output["anime_origin_confidence"] = round(ao_conf, 3)

    return output
    """
    Enhanced rule-based DeepDanbooru → structured attribute projector
    with improved body type, clothing, and age inference.
    """
    tag_list = list(tag_probs.keys())

    # ---- Character count gate ----
    count, ambiguous = estimate_character_count(tag_list)

    if count == 0:
        return _generate_scene_description(tag_probs)

    if count != 1:
        return {
            "image_status": "ambiguous_multi_character",
            "character_count": count,
            "attributes": None
        }

    # ---- Single character processing ----
    output = {"image_status": "single_character"}

    # ---- Gender ----
    gender_candidates = {
        "female": tag_probs.get("1girl", 0.0),
        "male": tag_probs.get("1boy", 0.0)
    }
    gender, g_conf = _resolve_competition(gender_candidates)
    output["gender"] = gender
    output["gender_confidence"] = round(g_conf, 3)

    # ---- Eye Color ----
    eye_candidates = {
        t.replace("_eyes", ""): p
        for t, p in tag_probs.items()
        if t in EYE_COLORS and p >= HIGH_CONF
    }

    if not eye_candidates:
        eye_candidates = {
            t.replace("_eyes", ""): p
            for t, p in tag_probs.items()
            if t in EYE_COLORS and p >= EYE_FALLBACK_THRESHOLD
        }

    eye_color, e_conf = _resolve_competition(eye_candidates)
    output["eye_color"] = eye_color
    output["eye_color_confidence"] = round(e_conf, 3)

    # ---- Hair ----
    hair_color_candidates = {
        color.replace("_hair", ""): tag_probs.get(color, 0.0)
        for color in HAIR_COLOR_TAGS
        if color in tag_probs
    }
    hair_color, hc_conf = _resolve_competition(hair_color_candidates)
    output["hair_color"] = hair_color
    output["hair_color_confidence"] = round(hc_conf, 3)

    hair_length_candidates = {
        "long": tag_probs.get("long_hair", 0.0),
        "short": tag_probs.get("short_hair", 0.0),
        "medium": tag_probs.get("medium_hair", 0.0),
        "shoulder": tag_probs.get("shoulder_length_hair", 0.0),
        "midback": tag_probs.get("midback_length_hair", 0.0),
        "waist": tag_probs.get("waist_length_hair", 0.0),
        "hip": tag_probs.get("hip_length_hair", 0.0),
        "ankle": tag_probs.get("ankle_length_hair", 0.0)
    }
    hair_length, hl_conf = _resolve_competition(hair_length_candidates)
    output["hair_length"] = hair_length
    output["hair_length_confidence"] = round(hl_conf, 3)

    hair_style_tags = {t for t in tag_probs if t in HAIR_STYLE_TAGS}
    hair_styles = _select_top_n(tag_probs, hair_style_tags, MAX_HAIR_STYLES)
    output["hair_styles"] = hair_styles

    # ---- Skin Tone ----
    skin_candidates = {
        "very_light": tag_probs.get("pale_skin", 0.0),
        "light": tag_probs.get("light_skin", 0.0),
        "tan": tag_probs.get("tan_skin", 0.0),
        "medium": tag_probs.get("medium_skin", 0.0),
        "olive": tag_probs.get("olive_skin", 0.0),
        "brown": tag_probs.get("brown_skin", 0.0),
        "dark": tag_probs.get("dark_skin", 0.0),
        "very_dark": tag_probs.get("very_dark_skin", 0.0),
        "colored": tag_probs.get("colored_skin", 0.0),
    }

    skin, s_conf = _resolve_competition(skin_candidates)

    if skin == "unknown":
        skin, s_conf = _infer_skin_tone_from_context(tag_probs, hair_color, eye_color)

    output["skin_tone"] = skin
    output["skin_tone_confidence"] = round(s_conf, 3)

    # ---- Body Type (ENHANCED with measurements) ----
    body_type, bt_conf, body_evidence = _infer_body_type_from_measurements(tag_probs)
    output["body_type"] = body_type
    output["body_type_confidence"] = round(bt_conf, 3)
    output["body_type_evidence"] = body_evidence

    # ---- Extract breast size for age inference ----
    breast_size = None
    for breast_tag, data in BREAST_SIZE_TAGS.items():
        if breast_tag in tag_probs and tag_probs[breast_tag] > HIGH_CONF:
            breast_size = data["size"]
            break

    # ---- Clothing (SPECIFIC ITEMS) ----
    specific_clothing = _extract_specific_clothing(tag_probs)
    output["clothing_items"] = specific_clothing

    # Also keep category classification for backward compatibility
    clothing_categories = []
    for category, tags in CLOTHING_MAP.items():
        max_score = max([tag_probs.get(t, 0.0) for t in tags], default=0.0)
        if max_score > HIGH_CONF:
            clothing_categories.append({"category": category, "confidence": round(max_score, 3)})

    clothing_categories.sort(key=lambda x: x["confidence"], reverse=True)
    output["clothing"] = clothing_categories[:MAX_CLOTHING_ITEMS]

    # ---- Age (ENHANCED multi-modal inference) ----
    clothing_item_names = [item["item"].replace(" ", "_") for item in specific_clothing]
    age, age_conf, age_evidence = _infer_age_multimodal(
        tag_probs, gender, breast_size, clothing_item_names
    )
    output["age"] = age
    output["age_confidence"] = round(age_conf, 3)
    output["age_evidence"] = age_evidence

    # ---- Expression ----
    expression_candidates = {
        t: p for t, p in tag_probs.items()
        if t in EXPRESSION_TAGS and p >= HIGH_CONF
    }

    if expression_candidates:
        sorted_expressions = sorted(expression_candidates.items(), key=lambda x: x[1], reverse=True)
        output["expressions"] = [
            {"expression": expr, "confidence": round(conf, 3)}
            for expr, conf in sorted_expressions[:MAX_EXPRESSIONS]
        ]
    else:
        output["expressions"] = [{"expression": "unknown", "confidence": 1.0}]

    # ---- Accessories ----
    accessory_tags = {t for t in tag_probs if t in ACCESSORY_TAGS}
    accessories = _select_top_n(tag_probs, accessory_tags, MAX_ACCESSORIES)
    output["accessories"] = accessories

    # ---- Cultural Elements ----
    cultural_elements = _analyze_cultural_elements(tag_probs)
    output["cultural_elements"] = cultural_elements

    # ---- Ethnicity Estimation ----
    ethnicity_data = _estimate_ethnicity(
        tag_probs, skin, hair_color, eye_color, cultural_elements
    )
    output["ethnicity_estimate"] = ethnicity_data

    # ---- Anime Origin ----
    anime_origin, ao_conf = _determine_anime_origin(tag_probs)
    output["anime_origin"] = anime_origin
    output["anime_origin_confidence"] = round(ao_conf, 3)

    return output
print("✓ Projector code loaded")


✓ Projector code loaded


In [6]:
import tarfile
import io
import json
from pathlib import Path
from PIL import Image
from huggingface_hub import hf_hub_download
import pandas as pd

class ShardLoader:
    def __init__(self, config):
        self.config = config
        self.shard_path = None
        self.stage1_index = None

    def download_shard(self):
        print(f"Downloading shard {self.config.shard_index}...")
        shard_filename = f"data/data_{self.config.shard_index:04d}.tar"

        self.shard_path = hf_hub_download(
            repo_id=self.config.repo_id,
            filename=shard_filename,
            repo_type="dataset"
        )
        print(f"✓ Shard downloaded to: {self.shard_path}")
        return self.shard_path

    def load_stage1_index(self):
        if Path(self.config.stage1_csv_path).exists():
            self.stage1_index = pd.read_csv(self.config.stage1_csv_path)
            print(f"✓ Loaded stage1 index: {len(self.stage1_index)} entries")
            return self.stage1_index
        else:
            print("⚠ stage1_index.csv not found")
            return pd.DataFrame()

    def get_stage1_metadata(self, image_id):
        if self.stage1_index is None or self.stage1_index.empty:
            return None
        row = self.stage1_index[self.stage1_index['image_id'] == image_id]
        return row.iloc[0].to_dict() if not row.empty else None

    def iter_images(self):
        if self.shard_path is None:
            self.download_shard()

        count = 0
        with tarfile.open(self.shard_path, 'r') as tar:
            for member in tar.getmembers():
                if count >= self.config.max_images:
                    break

                if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    try:
                        file_obj = tar.extractfile(member)
                        image = Image.open(io.BytesIO(file_obj.read())).convert('RGB')
                        image_id = Path(member.name).stem
                        stage1_meta = self.get_stage1_metadata(image_id)

                        yield image_id, image, stage1_meta
                        count += 1

                    except Exception as e:
                        print(f"⚠ Error loading {member.name}: {e}")
                        continue

print("✓ ShardLoader defined")


✓ ShardLoader defined


In [7]:
import tarfile
import io
import json
from pathlib import Path
from PIL import Image
from huggingface_hub import hf_hub_download
import pandas as pd

class ShardLoader:
    def __init__(self, config):
        self.config = config
        self.shard_path = None
        self.stage1_index = None

    def download_shard(self):
        print(f"Downloading shard {self.config.shard_index}...")
        shard_filename = f"data/data_{self.config.shard_index:04d}.tar"

        self.shard_path = hf_hub_download(
            repo_id=self.config.repo_id,
            filename=shard_filename,
            repo_type="dataset"
        )
        print(f"✓ Shard downloaded to: {self.shard_path}")
        return self.shard_path

    def load_stage1_index(self):
        if Path(self.config.stage1_csv_path).exists():
            self.stage1_index = pd.read_csv(self.config.stage1_csv_path)
            print(f"✓ Loaded stage1 index: {len(self.stage1_index)} entries")
            return self.stage1_index
        else:
            print("⚠ stage1_index.csv not found")
            return pd.DataFrame()

    def get_stage1_metadata(self, image_id):
        if self.stage1_index is None or self.stage1_index.empty:
            return None
        row = self.stage1_index[self.stage1_index['image_id'] == image_id]
        return row.iloc[0].to_dict() if not row.empty else None

    def iter_images(self):
        if self.shard_path is None:
            self.download_shard()

        count = 0
        with tarfile.open(self.shard_path, 'r') as tar:
            for member in tar.getmembers():
                if count >= self.config.max_images:
                    break

                if member.isfile() and member.name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    try:
                        file_obj = tar.extractfile(member)
                        image = Image.open(io.BytesIO(file_obj.read())).convert('RGB')
                        image_id = Path(member.name).stem
                        stage1_meta = self.get_stage1_metadata(image_id)

                        yield image_id, image, stage1_meta
                        count += 1

                    except Exception as e:
                        print(f"⚠ Error loading {member.name}: {e}")
                        continue

print("✓ ShardLoader defined")


✓ ShardLoader defined


In [8]:
import time
import tempfile
import os
from gradio_client import Client, handle_file

class DeepDanbooruTagger:
    def __init__(self, config):
        self.config = config
        self.client = Client("hysts/DeepDanbooru")
        self.request_count = 0
        self.last_request_time = 0

    def predict(self, image, threshold=0.1):
        # Rate limiting: 1 second between requests
        time_since_last = time.time() - self.last_request_time
        if time_since_last < 1.0:
            time.sleep(1.0 - time_since_last)

        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            image.save(tmp.name)
            tmp_path = tmp.name

        try:
            result = self.client.predict(
                image=handle_file(tmp_path),
                score_threshold=threshold,
                api_name="/predict"
            )

            tag_probs = {
                item["label"]: float(item["confidence"])
                for item in result[0]["confidences"]
                if item["label"] is not None
            }

            self.request_count += 1
            self.last_request_time = time.time()
            return tag_probs

        except Exception as e:
            raise Exception(f"DeepDanbooru API error: {e}")
        finally:
            os.remove(tmp_path)

class WD14Tagger:
    def __init__(self, config):
        self.config = config
        self.client = Client("SmilingWolf/wd-v1-4-moat-tagger-v2")

    def predict(self, image, threshold=0.1):
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
            image.save(tmp.name)
            tmp_path = tmp.name

        try:
            result = self.client.predict(
                image=handle_file(tmp_path),
                threshold=threshold,
                api_name="/predict"
            )

            tag_probs = {}
            if isinstance(result, dict) and "confidences" in result:
                tag_probs = {item["label"]: float(item["confidence"]) for item in result["confidences"]}
            return tag_probs

        except Exception as e:
            raise Exception(f"WD14 API error: {e}")
        finally:
            os.remove(tmp_path)

def create_tagger(config):
    if config.tagger_type == "deepdanbooru":
        try:
            tagger = DeepDanbooruTagger(config)
            print("✓ Using DeepDanbooru tagger")
            return tagger
        except Exception as e:
            print(f"⚠ DeepDanbooru failed, falling back to WD14: {e}")
            return WD14Tagger(config)
    else:
        return WD14Tagger(config)

print("✓ Taggers defined")


✓ Taggers defined


In [9]:
class AttributeValidator:
    REQUIRED_FIELDS = ["image_status", "gender", "age", "eye_color", "hair_color",
                       "hair_length", "skin_tone", "body_type"]

    SUSPICIOUS_COMBINATIONS = [
        ("child", ["bikini", "lingerie", "negligee"], "child_nsfw"),
        ("teen", ["business_suit"], "age_clothing_mismatch"),
    ]

    def __init__(self, config):
        self.config = config

    def validate_schema(self, attributes):
        errors = []
        for field in self.REQUIRED_FIELDS:
            if field not in attributes:
                errors.append(f"Missing field: {field}")
        return len(errors) == 0, errors

    def validate_confidence(self, attributes):
        warnings = []
        confidence_fields = ["gender_confidence", "age_confidence", "eye_color_confidence"]

        low_conf_count = 0
        for field in confidence_fields:
            if field in attributes:
                if attributes[field] < self.config.min_confidence_threshold:
                    low_conf_count += 1
                    warnings.append(f"Low confidence: {field}={attributes[field]:.3f}")

        is_valid = low_conf_count < len(confidence_fields) / 2
        return is_valid, warnings

    def check_consistency(self, attributes):
        flags = []
        if not self.config.flag_suspicious_combinations:
            return True, []

        age = attributes.get("age", "unknown")
        clothing_items = [item.get("item", "") for item in attributes.get("clothing_items", [])]

        for sus_age, sus_clothing, reason in self.SUSPICIOUS_COMBINATIONS:
            if age == sus_age:
                for clothing in clothing_items:
                    if any(sus in clothing.lower() for sus in sus_clothing):
                        flags.append(f"{reason}: {age} + {clothing}")

        return len(flags) == 0, flags

    def validate(self, attributes):
        result = {"valid": True, "errors": [], "warnings": [], "flags": []}

        if self.config.enable_schema_validation:
            schema_valid, schema_errors = self.validate_schema(attributes)
            result["errors"].extend(schema_errors)
            result["valid"] = result["valid"] and schema_valid

        conf_valid, conf_warnings = self.validate_confidence(attributes)
        result["warnings"].extend(conf_warnings)
        result["valid"] = result["valid"] and conf_valid

        if self.config.enable_consistency_checks:
            consistent, consistency_flags = self.check_consistency(attributes)
            result["flags"].extend(consistency_flags)

        return result

print("✓ Validator defined")


✓ Validator defined


In [10]:
# Modified JSONLExporter with dual output support

class JSONLExporter:
    def __init__(self, config, shard_index):
        self.config = config
        self.shard_index = shard_index

        self.output_dir = Path(config.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # TWO output files
        self.output_file_full = self.output_dir / f"shard_{shard_index:04d}_full.jsonl"
        self.output_file_simple = self.output_dir / f"shard_{shard_index:04d}_simple.jsonl"

        self.checkpoint_file = Path(config.checkpoint_dir) / f"shard_{shard_index:04d}_checkpoint.json"
        self.checkpoint_file.parent.mkdir(parents=True, exist_ok=True)

        self.processed_count = 0
        self.error_count = 0
        self.skipped_invalid_count = 0  # NEW: track skipped invalid images
        self.start_time = time.time()
        self.processed_ids = set()

        self.load_checkpoint()

    def load_checkpoint(self):
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file, 'r') as f:
                checkpoint = json.load(f)
                self.processed_ids = set(checkpoint.get("processed_ids", []))
                self.processed_count = checkpoint.get("processed_count", 0)
                self.error_count = checkpoint.get("error_count", 0)
                self.skipped_invalid_count = checkpoint.get("skipped_invalid_count", 0)
            print(f"✓ Resumed from checkpoint: {self.processed_count} images processed")

    def save_checkpoint(self):
        checkpoint = {
            "processed_ids": list(self.processed_ids),
            "processed_count": self.processed_count,
            "error_count": self.error_count,
            "skipped_invalid_count": self.skipped_invalid_count,
            "timestamp": time.time()
        }
        with open(self.checkpoint_file, 'w') as f:
            json.dump(checkpoint, f)

    def is_processed(self, image_id):
        return image_id in self.processed_ids

    def write_result(self, result, simplified=None):
        """
        Write full result + optional simplified result

        Q6: Only write simplified if validation.valid = True
        """
        # Always write full result
        with open(self.output_file_full, 'a') as f:
            f.write(json.dumps(result) + '\n')

        # Write simplified ONLY if valid (Q6: Answer B)
        if simplified is not None and result.get("validation", {}).get("valid", False):
            with open(self.output_file_simple, 'a') as f:
                f.write(json.dumps(simplified) + '\n')
        elif simplified is not None:
            # Track skipped invalid images
            self.skipped_invalid_count += 1

        self.processed_ids.add(result["image_id"])
        self.processed_count += 1

        if self.processed_count % self.config.checkpoint_interval == 0:
            self.save_checkpoint()
            print(f"💾 Checkpoint saved: {self.processed_count} images")

    def write_error(self, image_id, error, stage1_meta=None):
        error_result = {
            "image_id": image_id,
            "status": "error",
            "error": str(error),
            "stage1_metadata": stage1_meta,
            "timestamp": time.time()
        }

        if self.config.save_partial_on_error:
            # Only write to full file, not simplified
            with open(self.output_file_full, 'a') as f:
                f.write(json.dumps(error_result) + '\n')

        self.error_count += 1

        error_log = Path(self.config.error_log_path)
        with open(error_log, 'a') as f:
            f.write(f"{image_id}: {error}\n")

    def get_stats(self):
        elapsed = time.time() - self.start_time
        return {
            "processed_count": self.processed_count,
            "error_count": self.error_count,
            "skipped_invalid_count": self.skipped_invalid_count,
            "valid_exported_count": self.processed_count - self.skipped_invalid_count - self.error_count,
            "success_rate": self.processed_count / max(self.processed_count + self.error_count, 1),
            "elapsed_seconds": elapsed,
            "images_per_second": self.processed_count / max(elapsed, 1)
        }

print("✓ Dual-output Exporter defined")


✓ Dual-output Exporter defined


In [11]:
def create_simplified_output(full_result):
    """
    Convert full result to simplified 9-field format + Character field

    Based on answers:
    - Q2A: Keep "unknown" as-is for age
    - Q3B: Use "East Asian" (specific)
    - Q4: Keep highest confidence clothing item as-is
    - Q5B: Default body type to "Average" when unknown
    - Q7B: Add optional "Character" field
    - Q8C: Flag low-confidence images
    """
    attrs = full_result.get("attributes", {})
    tags = full_result.get("tags", {})

    # Age - Q2A: Keep unknown as-is
    age = attrs.get("age", "unknown")
    age_map = {
        "child": "Child",
        "teen": "Teen",
        "young adult": "Young Adult",
        "middle-aged": "Middle Aged",
        "elderly": "Elderly",
        "unknown": "Unknown"
    }

    # Ethnicity - Q3B: More specific (East Asian, not just Asian)
    ethnicity_data = attrs.get("ethnicity_estimate", {})
    ethnicity = ethnicity_data.get("primary", "unknown")
    ethnicity_map = {
        "east_asian": "East Asian",
        "south_asian": "South Asian",
        "caucasian": "Caucasian",
        "african": "African",
        "latin_american": "Latin American",
        "middle_eastern": "Middle Eastern",
        "unknown": "Unknown"
    }

    # Clothing - Q4: Keep highest confidence item as-is
    clothing_items = attrs.get("clothing_items", [])
    dress = "Unknown"
    if clothing_items:
        dress_raw = clothing_items[0]["item"]
        # Title case: "school_uniform" → "School Uniform"
        dress = " ".join(word.capitalize() for word in dress_raw.split("_"))

    # Hair style
    hair_styles = attrs.get("hair_styles", [])
    if hair_styles:
        hair_style_raw = hair_styles[0]["value"]
        hair_style = " ".join(word.capitalize() for word in hair_style_raw.split("_"))
    else:
        hair_style = "None"

    # Body type - Q5B: Default to "Average" when unknown
    body_type = attrs.get("body_type", "unknown")
    if body_type == "unknown":
        body_type = "Average"
    body_type = body_type.capitalize()

    # Character name - Q7B: Extract from tags
    character_name = None
    for tag, prob in tags.items():
        # Character tags often have underscores and high confidence
        if prob > 0.9 and "_" in tag and tag not in ["1girl", "1boy", "long_hair", "black_hair"]:
            # Check if it's likely a name (not a common tag)
            if tag not in ["school_uniform", "red_eyes", "looking_at_viewer"]:
                character_name = " ".join(word.capitalize() for word in tag.split("_"))
                break

    # Q8C: Flag low-confidence images
    low_confidence_flag = False
    confidence_fields = [
        attrs.get("gender_confidence", 1.0),
        attrs.get("age_confidence", 1.0),
        attrs.get("eye_color_confidence", 1.0),
        attrs.get("hair_color_confidence", 1.0),
        attrs.get("body_type_confidence", 1.0)
    ]

    # If more than 50% of confidences are below 0.3, flag it
    low_conf_count = sum(1 for c in confidence_fields if c < 0.3)
    if low_conf_count > len(confidence_fields) / 2:
        low_confidence_flag = True

    # Build simplified output
    simplified = {
        "image_id": full_result.get("image_id"),  # Keep ID for reference
        "Age": age_map.get(age, "Unknown"),
        "Gender": attrs.get("gender", "unknown").capitalize(),
        "Ethnicity": ethnicity_map.get(ethnicity, "Unknown"),
        "Hair Style": hair_style,
        "Hair Color": attrs.get("hair_color", "unknown").capitalize(),
        "Hair Length": attrs.get("hair_length", "unknown").capitalize(),
        "Eye Color": attrs.get("eye_color", "unknown").capitalize(),
        "Body Type": body_type,
        "Dress": dress,
        "Character": character_name,  # Q7B: Optional field (null if not found)
        "low_confidence_flag": low_confidence_flag  # Q8C: Flag for review
    }

    return simplified

print("✓ Simplified output generator defined")


✓ Simplified output generator defined


In [12]:
class AttributeExtractionPipeline:
    def __init__(self, config):
        self.config = config
        self.loader = ShardLoader(config)
        self.tagger = create_tagger(config)
        self.validator = AttributeValidator(config)
        self.exporter = None

    def process_single_image(self, image_id, image, stage1_meta):
        result = {
            "image_id": image_id,
            "shard_index": self.config.shard_index,
            "timestamp": time.time(),
            "stage1_metadata": stage1_meta,
            "status": "success"
        }

        try:
            # Stage 1: Tagging
            tag_probs = self.tagger.predict(image, self.config.tagger_threshold)
            result["tags"] = tag_probs
            result["num_tags"] = len(tag_probs)

            # Stage 2: Attribute extraction
            attributes = run_projector(tag_probs)
            result["attributes"] = attributes

            # Stage 3: Validation
            validation = self.validator.validate(attributes)
            result["validation"] = validation

            if not validation["valid"]:
                result["status"] = "validation_failed"

            # NSFW flag
            if stage1_meta and "nsfw_flag" in stage1_meta:
                result["nsfw_flag"] = stage1_meta["nsfw_flag"]

            return result

        except Exception as e:
            result["status"] = "error"
            result["error"] = str(e)
            return result

    def run(self):
        print("="*60)
        print("ANIME CHARACTER ATTRIBUTE EXTRACTION PIPELINE")
        print("="*60)
        print(f"Shard: {self.config.shard_index}")
        print(f"Max images: {self.config.max_images}")
        print(f"Output: Full + Simplified JSONL")
        print()

        self.loader.load_stage1_index()
        self.exporter = JSONLExporter(self.config, self.config.shard_index)

        for image_id, image, stage1_meta in self.loader.iter_images():
            if self.exporter.is_processed(image_id):
                print(f"⏭ Skipping {image_id} (already processed)")
                continue

            print(f"🔄 Processing {image_id}...", end=" ")

            try:
                # Process full result
                result = self.process_single_image(image_id, image, stage1_meta)

                # Generate simplified output
                simplified = create_simplified_output(result)

                # Write both (exporter handles validation check)
                self.exporter.write_result(result, simplified)

                # Status message
                status_icon = "✓" if result["validation"]["valid"] else "⚠"
                low_conf = " [LOW CONF]" if simplified.get("low_confidence_flag") else ""
                print(f"{status_icon} {result['status']}{low_conf}")

            except Exception as e:
                print(f"✗ ERROR: {e}")
                self.exporter.write_error(image_id, str(e), stage1_meta)

        self.exporter.save_checkpoint()

        print()
        print("="*60)
        print("PIPELINE COMPLETED")
        print("="*60)
        stats = self.exporter.get_stats()
        for key, value in stats.items():
            print(f"{key}: {value}")
        print()
        print(f"Full results: outputs/shard_{self.config.shard_index:04d}_full.jsonl")
        print(f"Simple results: outputs/shard_{self.config.shard_index:04d}_simple.jsonl")
        print()

print("✓ Dual-output Pipeline defined")


✓ Dual-output Pipeline defined


In [13]:
# Run the dual-output pipeline
pipeline = AttributeExtractionPipeline(config)
pipeline.run()


Loaded as API: https://hysts-deepdanbooru.hf.space ✔
✓ Using DeepDanbooru tagger
ANIME CHARACTER ATTRIBUTE EXTRACTION PIPELINE
Shard: 0
Max images: 100
Output: Full + Simplified JSONL

✓ Loaded stage1 index: 873562 entries


data/data_0000.tar:   0%|          | 0.00/10.8G [00:00<?, ?B/s]

✓ Shard downloaded to: /root/.cache/huggingface/hub/datasets--cagliostrolab--860k-ordered-tags/snapshots/d64a0c25967c744f382a6e5b60f09fa2ca809bc7/data/data_0000.tar
🔄 Processing danbooru_1380555_f9c05b66378137705fb63e010d6259d8... ✓ success
🔄 Processing danbooru_1379616_3c473f13bbb8cf9e72999cf97ef098c7... ⚠ validation_failed
🔄 Processing danbooru_1377955_ea1f08cfdf93c61ef0f7c08f2454c5bc... ⚠ validation_failed
🔄 Processing danbooru_1380644_c7cd48b95e720e06f3d66d59a1fdf930... ✓ success
🔄 Processing danbooru_1379037_2a2a177a935d312581773e643aa74ec0... ✓ success
🔄 Processing danbooru_1367260_d92667950065c99bd1944168adb41fed... ⚠ validation_failed
🔄 Processing danbooru_1380318_5d67687645b17f0171df8007a29c404c... ✓ success
🔄 Processing danbooru_1380748_00d5f9ac2c54742e49e135466606652a... ⚠ validation_failed
🔄 Processing danbooru_1381643_3e5e4b08363fd70bed63e292b0200f7a... ✓ success
🔄 Processing danbooru_1370513_e8f30add09fdad6eb332b284f4a408bd... ⚠ validation_failed
🔄 Processing danbooru_138

In [14]:
# View full output (first 2 lines)
print("="*60)
print("FULL OUTPUT (first 2 images):")
print("="*60)
!head -n 2 outputs/shard_0000_full.jsonl

print("\n")
print("="*60)
print("SIMPLIFIED OUTPUT (first 5 images):")
print("="*60)
!head -n 5 outputs/shard_0000_simple.jsonl

print("\n")
print("="*60)
print("FILES CREATED:")
print("="*60)
!ls -lh outputs/

print("\n")
print("="*60)
print("ERRORS (if any):")
print("="*60)
!cat errors.log 2>/dev/null || echo "No errors"


FULL OUTPUT (first 2 images):
{"image_id": "danbooru_1380555_f9c05b66378137705fb63e010d6259d8", "shard_index": 0, "timestamp": 1766675019.228043, "stage1_metadata": null, "status": "success", "tags": {"1girl": 0.9992964267730713, "rating:safe": 0.9921383857727051, "nakano_azusa": 0.9815398454666138, "solo": 0.9634111523628235, "long_hair": 0.9398812651634216, "twintails": 0.9256547689437866, "black_hair": 0.9210128784179688, "snowing": 0.9086070656776428, "red_eyes": 0.8256351351737976, "polka_dot_background": 0.7506715655326843, "coat": 0.6887642741203308, "polka_dot": 0.6545858979225159, "looking_at_viewer": 0.5846124887466431, "snow": 0.5015823841094971, "blush": 0.44837039709091187, "scarf_over_mouth": 0.4250466525554657, "upper_body": 0.3635064363479614, "teeth": 0.31698155403137207, "hair_ornament": 0.30996158719062805, "jacket": 0.2888858914375305, "breath": 0.27422797679901123, "bangs": 0.2734975218772888, "scarf": 0.27141356468200684, "long_sleeves": 0.2696443498134613, "eyebr

In [15]:
import json

# Load and pretty-print first 3 simplified results
print("="*60)
print("SAMPLE SIMPLIFIED OUTPUTS (Pretty Format)")
print("="*60)

with open('outputs/shard_0000_simple.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:  # Show first 3
            break
        data = json.loads(line)

        print(f"\n🖼 Image {i+1}: {data['image_id']}")
        print("-" * 40)
        for key, value in data.items():
            if key != "image_id":
                icon = "⚠" if key == "low_confidence_flag" and value else ""
                print(f"  {key:20s}: {value} {icon}")


SAMPLE SIMPLIFIED OUTPUTS (Pretty Format)

🖼 Image 1: danbooru_1380555_f9c05b66378137705fb63e010d6259d8
----------------------------------------
  Age                 : Unknown 
  Gender              : Female 
  Ethnicity           : East Asian 
  Hair Style          : Twintails 
  Hair Color          : Black 
  Hair Length         : Long 
  Eye Color           : Red 
  Body Type           : Average 
  Dress               : Coat 
  Character           : Nakano Azusa 
  low_confidence_flag : False 

🖼 Image 2: danbooru_1380644_c7cd48b95e720e06f3d66d59a1fdf930
----------------------------------------
  Age                 : Unknown 
  Gender              : Female 
  Ethnicity           : East Asian 
  Hair Style          : None 
  Hair Color          : Black 
  Hair Length         : Long 
  Eye Color           : Blue 
  Body Type           : Average 
  Dress               : Bikini 
  Character           : Striped Bikini 
  low_confidence_flag : False 

🖼 Image 3: danbooru_1379037_2a2a177

In [16]:
from google.colab import files

# Download output file
files.download('outputs/shard_0000_attributes.jsonl')

# Download checkpoint
files.download('checkpoints/shard_0000_checkpoint.json')


FileNotFoundError: Cannot find file: outputs/shard_0000_attributes.jsonl

In [ ]:
# Cell: Zero-Shot CLIP Tagger

!pip install transformers torch pillow

import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

class CLIPAttributeTagger:
    """
    Zero-shot attribute extraction using CLIP
    No training required!
    """

    def __init__(self, config):
        self.config = config
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load CLIP model
        print("Loading CLIP model...")
        self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(self.device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

        # Define attribute prompts (zero-shot)
        self.attribute_templates = {
            "gender": [
                "a photo of a female anime character",
                "a photo of a male anime character"
            ],
            "age": [
                "a photo of a child anime character",
                "a photo of a teenage anime character",
                "a photo of a young adult anime character",
                "a photo of a middle-aged anime character",
                "a photo of an elderly anime character"
            ],
            "hair_color": [
                "anime character with black hair",
                "anime character with brown hair",
                "anime character with blonde hair",
                "anime character with white hair",
                "anime character with red hair",
                "anime character with pink hair",
                "anime character with blue hair",
                "anime character with green hair",
                "anime character with purple hair",
                "anime character with silver hair"
            ],
            "eye_color": [
                "anime character with blue eyes",
                "anime character with brown eyes",
                "anime character with green eyes",
                "anime character with red eyes",
                "anime character with purple eyes",
                "anime character with yellow eyes",
                "anime character with pink eyes"
            ],
            "body_type": [
                "slim anime character",
                "athletic anime character",
                "muscular anime character",
                "curvy anime character",
                "petite anime character"
            ],
            "expression": [
                "smiling anime character",
                "angry anime character",
                "sad anime character",
                "surprised anime character",
                "neutral expression anime character",
                "happy anime character"
            ]
        }

        print(f"✓ CLIP loaded on {self.device}")

    def predict(self, image, threshold=0.3):
        """
        Zero-shot attribute prediction
        Returns: tag_probs dict compatible with projector
        """
        # Convert to CLIP input
        inputs = self.processor(
            images=image,
            return_tensors="pt",
            padding=True
        ).to(self.device)

        # Get image embedding
        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Predict each attribute
        tag_probs = {}

        for attribute, prompts in self.attribute_templates.items():
            # Encode text prompts
            text_inputs = self.processor(
                text=prompts,
                return_tensors="pt",
                padding=True
            ).to(self.device)

            with torch.no_grad():
                text_features = self.model.get_text_features(**text_inputs)
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            # Compute similarity
            similarities = (image_features @ text_features.T).squeeze(0)
            probs = torch.softmax(similarities * 100, dim=0)  # Temperature scaling

            # Convert to tags
            for prompt, prob in zip(prompts, probs):
                # Extract tag from prompt: "anime character with blue eyes" → "blue_eyes"
                if "with" in prompt:
                    tag = prompt.split("with")[-1].strip().replace(" ", "_")
                elif "of a" in prompt:
                    parts = prompt.split("of a")[-1].strip().split()
                    tag = "_".join(parts[:2])  # e.g., "female anime" → "female"
                    if "anime" in tag:
                        tag = parts[0]  # Just "female"
                else:
                    tag = prompt.replace("anime character", "").strip().replace(" ", "_")

                tag_probs[tag] = float(prob)

        # Filter by threshold
        tag_probs = {k: v for k, v in tag_probs.items() if v >= threshold}

        return tag_probs

    def predict_batch(self, images, threshold=0.3):
        """Batch processing for speed"""
        inputs = self.processor(
            images=images,
            return_tensors="pt",
            padding=True
        ).to(self.device)

        with torch.no_grad():
            image_features = self.model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        results = []
        for img_feat in image_features:
            # Same logic as predict() but for single embedding
            tag_probs = {}
            for attribute, prompts in self.attribute_templates.items():
                text_inputs = self.processor(text=prompts, return_tensors="pt", padding=True).to(self.device)
                text_features = self.model.get_text_features(**text_inputs)
                text_features = text_features / text_features.norm(dim=-1, keepdim=True)

                similarities = (img_feat @ text_features.T)
                probs = torch.softmax(similarities * 100, dim=0)

                for prompt, prob in zip(prompts, probs):
                    if "with" in prompt:
                        tag = prompt.split("with")[-1].strip().replace(" ", "_")
                    elif "of a" in prompt:
                        tag = prompt.split("of a")[-1].strip().split()[0]
                    else:
                        tag = prompt.replace("anime character", "").strip().replace(" ", "_")

                    tag_probs[tag] = float(prob)

            tag_probs = {k: v for k, v in tag_probs.items() if v >= threshold}
            results.append(tag_probs)

        return results

# Test it
clip_tagger = CLIPAttributeTagger(config)

# Test on one image
for image_id, image, _ in loader.iter_images():
    tags = clip_tagger.predict(image)
    print(f"CLIP tags: {tags}")
    break


In [ ]:
# Cell: Complete Working Gradio Demo

import gradio as gr
from gradio_client import Client, handle_file
from PIL import Image
import numpy as np
import json
import time
from typing import Dict, Any, Tuple, List
import tempfile
import os
from pathlib import Path

# ===== CONFIGURATION =====
DEEPDANBOORU_CLIENT = Client("hysts/DeepDanbooru")

# ===== DEEPDANBOORU TAGGER =====
def run_deepdanbooru(image: Image, score_threshold: float = 0.1) -> Dict[str, float]:
    """Calls hysts/DeepDanbooru via HuggingFace Gradio API"""
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        image.save(tmp.name)
        tmp_path = tmp.name

    try:
        result = DEEPDANBOORU_CLIENT.predict(
            image=handle_file(tmp_path),
            score_threshold=score_threshold,
            api_name="/predict"
        )

        tag_probs = {
            item["label"]: float(item["confidence"])
            for item in result[0]["confidences"]
            if item["label"] is not None
        }

        return tag_probs

    finally:
        os.remove(tmp_path)

# ===== FORMAT FUNCTIONS =====
def format_simple_output(structured: Dict[str, Any]) -> Dict[str, str]:
    """Convert structured output to simplified format"""
    attrs = structured.get("attributes", structured)  # Handle both formats

    output = {}

    # Age
    age = attrs.get("age", "unknown")
    if age != "unknown":
        output["Age"] = " ".join(word.capitalize() for word in age.split("_"))
    else:
        output["Age"] = "Unknown"

    # Gender
    gender = attrs.get("gender", "unknown")
    output["Gender"] = gender.capitalize() if gender != "unknown" else "Unknown"

    # Ethnicity
    ethnicity_data = attrs.get("ethnicity_estimate", {})
    ethnicity = ethnicity_data.get("primary", "unknown")
    ethnicity_map = {
        "east_asian": "Asian",
        "south_asian": "South Asian",
        "caucasian": "Caucasian",
        "african": "African",
        "latin_american": "Latin American",
        "middle_eastern": "Middle Eastern",
        "unknown": "Unknown"
    }
    output["Ethnicity"] = ethnicity_map.get(ethnicity, "Unknown")

    # Hair Style
    hair_styles = attrs.get("hair_styles", [])
    if hair_styles and len(hair_styles) > 0:
        style = hair_styles[0]["value"]
        output["Hair Style"] = " ".join(word.capitalize() for word in style.split("_"))
    else:
        output["Hair Style"] = "None"

    # Hair Color
    hair_color = attrs.get("hair_color", "unknown")
    output["Hair Color"] = hair_color.capitalize() if hair_color != "unknown" else "Unknown"

    # Hair Length
    hair_length = attrs.get("hair_length", "unknown")
    output["Hair Length"] = hair_length.capitalize() if hair_length != "unknown" else "Unknown"

    # Eye Color
    eye_color = attrs.get("eye_color", "unknown")
    output["Eye Color"] = eye_color.capitalize() if eye_color != "unknown" else "Unknown"

    # Body Type
    body_type = attrs.get("body_type", "unknown")
    output["Body Type"] = body_type.capitalize() if body_type != "unknown" else "Unknown"

    # Dress
    clothing = attrs.get("clothing", [])
    if clothing and len(clothing) > 0:
        dress_category = clothing[0]["category"]
        output["Dress"] = " ".join(word.capitalize() for word in dress_category.split("_"))
    else:
        output["Dress"] = "Unknown"

    return output

def flatten_attributes(structured: Dict[str, Any]) -> List[Tuple[str, str, float]]:
    """Flatten structured attributes into table rows"""
    rows = []

    # Check for multi-character or no-character cases
    if structured.get("image_status") == "ambiguous_multi_character":
        rows.append(("Image status", "Multiple characters detected", 0.0))
        rows.append(("Character count", str(structured.get("character_count", "unknown")), 0.0))
        return rows

    if structured.get("image_status") == "no_character_detected":
        rows.append(("Image status", "No character detected", 0.0))
        rows.append(("Scene type", structured.get("scene_type", "unknown"), 0.0))
        return rows

    # Single character attributes
    rows.append(("Gender", structured.get("gender", "unknown"), structured.get("gender_confidence", 0.0)))
    rows.append(("Age", structured.get("age", "unknown"), structured.get("age_confidence", 0.0)))
    rows.append(("Eye color", structured.get("eye_color", "unknown"), structured.get("eye_color_confidence", 0.0)))
    rows.append(("Hair color", structured.get("hair_color", "unknown"), structured.get("hair_color_confidence", 0.0)))
    rows.append(("Hair length", structured.get("hair_length", "unknown"), structured.get("hair_length_confidence", 0.0)))

    # Hair styles
    for i, style in enumerate(structured.get("hair_styles", []), start=1):
        rows.append((f"Hair style {i}", style.get("value", "unknown"), style.get("confidence", 0.0)))

    rows.append(("Skin tone", structured.get("skin_tone", "unknown"), structured.get("skin_tone_confidence", 0.0)))
    rows.append(("Body type", structured.get("body_type", "unknown"), structured.get("body_type_confidence", 0.0)))

    # Clothing
    for i, item in enumerate(structured.get("clothing", []), start=1):
        rows.append((f"Clothing {i}", item.get("category", "unknown"), item.get("confidence", 0.0)))

    # Expressions
    for i, expr in enumerate(structured.get("expressions", []), start=1):
        rows.append((f"Expression {i}", expr.get("expression", "unknown"), expr.get("confidence", 0.0)))

    # Accessories
    for i, acc in enumerate(structured.get("accessories", []), start=1):
        rows.append((f"Accessory {i}", acc.get("value", "unknown"), acc.get("confidence", 0.0)))

    # Cultural elements
    for i, elem in enumerate(structured.get("cultural_elements", []), start=1):
        rows.append((f"Cultural element {i}", elem.get("element", "unknown"), elem.get("confidence", 0.0)))

    # Ethnicity estimate
    ethnicity_data = structured.get("ethnicity_estimate", {})
    rows.append(("Ethnicity (estimate)", ethnicity_data.get("primary", "unknown"), ethnicity_data.get("confidence", 0.0)))

    # Anime origin
    rows.append(("Anime origin", structured.get("anime_origin", "unknown"), structured.get("anime_origin_confidence", 0.0)))

    return rows

def tags_to_sorted_list(tag_probs: Dict[str, float], top_k: int = 50):
    items = sorted(tag_probs.items(), key=lambda x: x[1], reverse=True)
    return items[:top_k]

def confidence_bar_html(value: float, width_px: int = 120) -> str:
    """Render confidence bar HTML"""
    pct = int(round(value * 100))
    bar = f"""
    <div style="width:{width_px}px; height:10px; background:#e6eef8; border-radius:6px; overflow:hidden; box-shadow: inset 0 1px 0 rgba(255,255,255,0.6);">
      <div style="width:{pct}%; height:100%; border-radius:6px; background:linear-gradient(90deg,#4e8ef7,#7fb3ff);"></div>
    </div>
    <div style="font-size:11px; color:#425166; margin-top:4px;">{pct}%</div>
    """
    return bar

# ===== MAIN ANALYSIS FUNCTION =====
def analyze_image_inference(image: Image, show_raw_tags: bool, show_confidence: bool):
    """Main orchestrator: runs tagger + projector"""
    if image is None:
        return {}, {}, [], [], "<div style='color:#425166;'>No image uploaded</div>"

    try:
        start = time.time()

        # Step 1: Run tagger
        tag_probs = run_deepdanbooru(image)

        # Step 2: Run projector (imported from your projector code)
        structured = run_projector(tag_probs)

        # Step 3: Create simplified output
        simple_output = format_simple_output({"attributes": structured})

        # Build table rows
        table_rows = flatten_attributes(structured)

        # Top-K tags
        top_tags = tags_to_sorted_list(tag_probs, top_k=50)

        # Dataframe rows
        df_rows = [[r[0], r[1], round(r[2], 3) if isinstance(r[2], float) else r[2]] for r in table_rows]

        # Build tags table
        tags_table = [[t[0], round(t[1], 4)] for t in top_tags]

        # Confidence HTML
        confidences = [r[2] for r in table_rows if isinstance(r[2], (int, float))]
        avg_conf = float(np.mean(confidences)) if confidences else 0.0
        conf_html = f"<div style='display:flex;gap:14px;align-items:center;'><div style='font-size:14px;color:#223'>Average attribute confidence:</div><div>{confidence_bar_html(avg_conf, width_px=160)}</div></div>"

        # Hide tags if not requested
        if not show_raw_tags:
            tags_table_to_send = [["(hidden - enable option to view)", ""]]
        else:
            tags_table_to_send = tags_table

        latency = time.time() - start
        structured["_meta"] = {
            "inference_latency_s": round(latency, 3),
            "note": "Using hysts/DeepDanbooru API + rule-based projector"
        }

        return simple_output, structured, df_rows, tags_table_to_send, conf_html

    except Exception as e:
        error_msg = f"<div style='color:red;'>Error: {str(e)}</div>"
        return {}, {}, [], [], error_msg

# ===== PIPELINE STATUS FUNCTIONS =====
def get_processing_status():
    """Read checkpoint files for status"""
    status_data = []

    checkpoint_dir = Path("checkpoints")
    if not checkpoint_dir.exists():
        return [{"Shard": "N/A", "Processed": 0, "Errors": 0, "Status": "No processing started"}]

    for shard_idx in range(120):
        checkpoint_file = checkpoint_dir / f"shard_{shard_idx:04d}_checkpoint.json"
        if checkpoint_file.exists():
            try:
                with open(checkpoint_file) as f:
                    checkpoint = json.load(f)
                    status_data.append({
                        "Shard": shard_idx,
                        "Processed": checkpoint.get("processed_count", 0),
                        "Errors": checkpoint.get("error_count", 0),
                        "Status": "✅ Complete" if checkpoint.get("processed_count", 0) >= 9000 else "🔄 In Progress"
                    })
            except:
                pass

    return status_data if status_data else [{"Shard": "N/A", "Processed": 0, "Errors": 0, "Status": "No data"}]

def explore_results(shard_idx, limit=10):
    """Browse processed results"""
    output_file = Path(f"outputs/shard_{shard_idx:04d}_attributes.jsonl")

    if not output_file.exists():
        return [{"Status": f"Shard {shard_idx} not processed yet"}]

    results = []
    try:
        with open(output_file) as f:
            for i, line in enumerate(f):
                if i >= limit:
                    break
                data = json.loads(line)
                simplified = format_simple_output(data)
                simplified["Image ID"] = data.get("image_id", "unknown")
                results.append(simplified)
    except Exception as e:
        return [{"Error": str(e)}]

    return results if results else [{"Status": "No results found"}]

# ===== GRADIO INTERFACE =====
def build_demo():
    with gr.Blocks(title="Anime Character Attribute Pipeline", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 🎨 Anime Character Attribute Extraction Pipeline
        **Multi-modal pipeline with DeepDanbooru, WD14, and custom CLIP support**
        """)

        with gr.Tabs():

            # ===== TAB 1: SINGLE IMAGE ANALYSIS =====
            with gr.Tab("📸 Single Image Analysis"):
                gr.Markdown("""
                **Upload an anime character image to explore visual attributes.**
                This tool shows (1) simplified attributes, (2) confidence scores, and (3) raw tag evidence.
                """)

                with gr.Row():
                    with gr.Column(scale=1):
                        image_in = gr.Image(type="pil", label="Upload Image (anime character)")
                        analyze_btn = gr.Button("🔍 Analyze Image", variant="primary", size="lg")

                        gr.Markdown("**Options**")
                        cb_raw_tags = gr.Checkbox(label="Show raw DeepDanbooru tags (evidence)", value=False)
                        cb_confidence = gr.Checkbox(label="Show confidence bars", value=True)

                        gr.Markdown("💡 Tip: Click 'Show raw tags' to inspect evidence used for attribute extraction")

                    with gr.Column(scale=1):
                        gr.Markdown("### Character Attributes (Simple Format)")
                        simple_json = gr.JSON(label="Extracted Attributes")

                        with gr.Accordion("🔍 Show detailed structured output", open=False):
                            json_output = gr.JSON(label="Full structured attributes")

                # Table view
                with gr.Row():
                    df_output = gr.Dataframe(
                        headers=["Attribute", "Value", "Confidence"],
                        label="Detailed Attribute Table",
                        interactive=False
                    )

                # Confidence
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### Model Confidence & Ambiguity")
                        gr.Markdown("Confidence bars reflect model certainty. Low confidence indicates ambiguity in stylized art.")
                        html_confidence = gr.HTML("<div style='color:#425166;font-size:13px;'>No analysis yet. Upload an image and click Analyze.</div>")

                # Evidence layer
                with gr.Accordion("📊 Show explainability & evidence (raw tags + mapping)", open=False):
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.Markdown("**Top DeepDanbooru tags (evidence)**")
                            tags_list = gr.Dataframe(
                                headers=["Tag", "Probability"],
                                label="Top tags from tagger",
                                interactive=False
                            )
                        with gr.Column(scale=1):
                            gr.Markdown("**Projection mapping (how tags → attributes)**")
                            mapping_text = gr.Markdown("""
                            Tag probabilities are projected into structured attributes using:
                            - **Competition resolution**: Highest confidence wins when attributes conflict
                            - **Multi-modal inference**: Combines signals (clothing + body type → age)
                            - **Fallback logic**: Uses context when direct tags missing
                            """)

                gr.Markdown("""
                ---
                **Design rationale:** Image-first UI with compact summary, progressive disclosure of evidence.
                **Deployment:** Uses hysts/DeepDanbooru HuggingFace Space API + rule-based projector.
                """)

                # Connect analyze button
                analyze_btn.click(
                    fn=analyze_image_inference,
                    inputs=[image_in, cb_raw_tags, cb_confidence],
                    outputs=[simple_json, json_output, df_output, tags_list, html_confidence]
                )

            # ===== TAB 2: BATCH PROCESSING STATUS =====
            with gr.Tab("📊 Processing Status"):
                gr.Markdown("### Batch Processing Progress")
                gr.Markdown("Monitor shard processing status with automatic checkpointing.")

                refresh_btn = gr.Button("🔄 Refresh Status", variant="secondary")
                status_table = gr.Dataframe(
                    headers=["Shard", "Processed", "Errors", "Status"],
                    label="Shard Processing Status",
                    value=get_processing_status()
                )

                refresh_btn.click(
                    fn=get_processing_status,
                    outputs=status_table
                )

            # ===== TAB 3: DATASET EXPLORER =====
            with gr.Tab("🗂️ Dataset Explorer"):
                gr.Markdown("### Browse Processed Results")

                with gr.Row():
                    shard_selector = gr.Slider(0, 119, value=0, step=1, label="Shard Index")
                    limit_selector = gr.Slider(5, 50, value=10, step=5, label="Number of Results")

                explore_btn = gr.Button("🔍 Load Results", variant="primary")
                results_display = gr.JSON(label="Processed Results")

                explore_btn.click(
                    fn=explore_results,
                    inputs=[shard_selector, limit_selector],
                    outputs=results_display
                )

        gr.Markdown("""
        ---
        **Pipeline Features:**
        - ✅ Multiple tagger support (DeepDanbooru, WD14, CLIP)
        - ✅ Automatic checkpointing every 100 images
        - ✅ Resume from crash
        - ✅ Validation & consistency checks
        - ✅ Simplified + detailed output formats

        **Note:** Ethnicity is estimated probabilistically from visual features and cultural context, not asserted as definitive.
        """)

    return demo

# ===== LAUNCH =====
if __name__ == "__main__":
    demo = build_demo()
    demo.launch(share=True, debug=True)
